# Multiagent Pipeline

In [28]:
# Facoltativo: installa dipendenze se ti servono in un ambiente pulito
# %pip install -r requirements.txt

# Configuration

In [29]:
import os
from dotenv import load_dotenv
import sys
import io
import traceback

load_dotenv()

LM_STUDIO_BASE_URL = "https://api.mistral.ai/v1"
LM_STUDIO_API_KEY  = os.getenv("MISTRAL_API_KEY", "")
LM_STUDIO_MODEL    = "mistral-small-latest"

# Paths 
PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
ALLARMI_CSV = os.path.join(RAW_DIR, "ALLARMI.csv")
TIPOLOGIA_CSV = os.path.join(RAW_DIR, "TIPOLOGIA_VIAGGIATORE.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "output")
FINDINGS_JSON = os.path.join(OUTPUT_DIR, "findings.json")
COLUMN_PROFILES_JSON = os.path.join(OUTPUT_DIR, "column_profiles.json")
CLEANING_PLAN_JSON   = os.path.join(OUTPUT_DIR, "cleaning_plan.json")   # NEW: slim JSON for cleaning agent

# Profiling 
DOMINANT_FORMAT_SAMPLE_SIZE = 200   # NEW: how many rows to inspect when detecting the dominant format

# Outlier Detection 
DEFAULT_OUTLIER_ALGORITHM = "IsolationForest"
ISOLATION_FOREST_CONTAMINATION = 0.05
LOF_N_NEIGHBORS = 20
ZSCORE_THRESHOLD = 3.0

# Risk Thresholds 
ALERT_RATE_MULTIPLIER = 3.0

In [30]:
# =============================================================================
# COLUMN ROLES
# =============================================================================
# Closed vocabulary of roles assignable to columns.
# The LLM picks one of these during profiling. If none fits, it can return
# "custom:<name>" to propose a new role.

COLUMN_ROLES = {
    # numeric
    "identifier_numeric",   # numeric IDs without arithmetic meaning (preserve as string to keep leading zeros)
    "count",                # non-negative integer counts (e.g. number of flights)
    "measure",              # continuous measurements (weight, distance, price)
    "percentage",           # percentage value
    "year",                 # year (e.g. 2024)
    "month_number",         # month as 01-12
    "day_number",           # day as 01-31

    # string / code
    "code_alphanumeric",    # mixed letters+digits codes (e.g. flight code "AZ123")
    "code_categorical",     # low-cardinality category (zone, type, ...)
    "country_code",         # ISO country code (IT, FR, ...)
    "country_name",         # full country name
    "city_name",            # city name
    "airport_code",         # IATA / ICAO airport code
    "person_name",          # person name
    "free_text",            # free text / operator notes
    "flag_binary",          # binary flag (yes/no, 0/1, alto/basso)

    # temporal
    "date",                 # date without time
    "datetime",             # date + time

    # special
    "unknown",              # LLM cannot classify
}

# Deterministic mapping role -> expected pandas dtype.
# The cleaning agent uses this AFTER applying transformations
# to enforce the final column dtype.

ROLE_TO_EXPECTED_DTYPE = {
    "identifier_numeric": "string",      # keep as string to preserve leading zeros
    "count":              "Int64",       # nullable integer
    "measure":            "Float64",     # nullable float
    "percentage":         "Float64",
    "year":               "Int64",
    "month_number":       "Int64",
    "day_number":         "Int64",

    "code_alphanumeric":  "string",
    "code_categorical":   "string",
    "country_code":       "string",
    "country_name":       "string",
    "city_name":          "string",
    "airport_code":       "string",
    "person_name":        "string",
    "free_text":          "string",
    "flag_binary":        "string",

    "date":               "datetime64[ns]",
    "datetime":           "datetime64[ns]",

    "unknown":            "string",      # safe default
}


def resolve_expected_dtype(role: str) -> str:
    """
    Resolve the expected pandas dtype for a given role.
    Handles 'custom:<name>' roles by defaulting to 'string'.
    """
    if role in ROLE_TO_EXPECTED_DTYPE:
        return ROLE_TO_EXPECTED_DTYPE[role]
    if isinstance(role, str) and role.startswith("custom:"):
        return "string"
    return "string"

In [31]:
# =============================================================================
# JSON HELPERS
# =============================================================================
# Utilities to convert pandas/numpy values into JSON-safe native types,
# and to load/write JSON files robustly.

import os
import re
import json
import math
from typing import Any, Callable, Optional

import numpy as np
import pandas as pd


def _to_native(value: Any) -> Any:
    """Convert pandas/numpy objects into native JSON-serializable Python types."""
    if isinstance(value, dict):
        return {str(k): _to_native(v) for k, v in value.items()}
    if isinstance(value, (np.ndarray, list, tuple)):
        return [_to_native(v) for v in value]
    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if math.isnan(float(value)):
            return None
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return value


def _safe_load_json(path: str) -> dict:
    """Load a JSON file safely; return an empty dict if missing/corrupted."""
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except Exception:
        return {}


def _safe_write_json(path: str, data: dict) -> None:
    """Write JSON deterministically with UTF-8 and 2-space indentation."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(_to_native(data), f, ensure_ascii=False, indent=2)

In [32]:
# =============================================================================
# MISSING VALUES HELPERS
# =============================================================================
# Detect both native NaN/None and "semantic" missing tokens (e.g. "n/a", "-").

DEFAULT_MISSING_TOKENS = {
    "", " ", "na", "n/a", "null", "none", "nan", "unknown", "undefined",
    "-", "--", "not available", "missing", "n.d.", "nd"
}


def _normalize_missing_mask(
    series: pd.Series,
    missing_tokens: Optional[set[str]] = None,
) -> pd.Series:
    """Return a boolean mask flagging missing values (native + semantic placeholders)."""
    missing_tokens = missing_tokens or DEFAULT_MISSING_TOKENS

    native_missing = series.isna()

    as_str = series.astype("string").str.strip().str.lower()
    semantic_missing = as_str.isin(missing_tokens)

    return native_missing | semantic_missing.fillna(False)


def _non_missing_series(
    series: pd.Series,
    missing_tokens: Optional[set[str]] = None,
) -> pd.Series:
    """Return only the non-missing values of a series."""
    mask = _normalize_missing_mask(series, missing_tokens=missing_tokens)
    return series.loc[~mask]


# =============================================================================
# DATAFRAME-LEVEL HELPERS
# =============================================================================

def get_dataframe_shape(df: pd.DataFrame) -> dict:
    """Return dataframe shape as {rows, columns}."""
    rows, cols = df.shape
    return {"rows": int(rows), "columns": int(cols)}


def get_dataframe_dtypes(df: pd.DataFrame) -> dict:
    """Return pandas dtype per column."""
    return {col: str(dtype) for col, dtype in df.dtypes.items()}


def get_dataframe_nulls(df: pd.DataFrame) -> dict:
    """Return null counts per column, including semantic missing tokens."""
    result = {}
    for col in df.columns:
        mask = _normalize_missing_mask(df[col])
        result[col] = {
            "null_count": int(mask.sum()),
            "non_null_count": int((~mask).sum()),
        }
    return result


def get_missing_percentages(df: pd.DataFrame) -> dict:
    """Return missing percentage per column."""
    total_rows = max(len(df), 1)
    result = {}
    for col in df.columns:
        mask = _normalize_missing_mask(df[col])
        result[col] = {
            "missing_percentage": round(float(mask.mean() * 100), 2),
            "missing_ratio": round(float(mask.sum() / total_rows), 4),
        }
    return result

In [33]:
# =============================================================================
# FORMAT DETECTION
# =============================================================================
# Lightweight format labeling: each value is classified into a coarse
# category (digit, alpha, alphanumeric, ...) used to detect the dominant
# value-format of a column.
#
# We deliberately keep the categories generic. Fine-grained semantic
# inference (is this a country code? a month number? a flight ID?) is
# delegated to the LLM via the role assignment.

def _value_format_label(value: Any) -> str:
    """Return a coarse format label for a single value."""
    if value is None:
        return "missing"

    s = str(value).strip()

    if s == "":
        return "empty"

    # Pure digits (e.g. "5", "12", "00123")
    if s.isdigit():
        return "digit"

    # Float-like (e.g. "12.34", "12,34", "-3.5")
    try:
        float(s.replace(",", "."))
        return "float"
    except Exception:
        pass

    # Pure letters (e.g. "gennaio", "italia")
    if s.isalpha():
        return "alpha"

    # Mixed letters + digits (e.g. "AZ123", "1 voli")
    has_alpha = any(c.isalpha() for c in s)
    has_digit = any(c.isdigit() for c in s)
    if has_alpha and has_digit:
        return "alphanumeric"

    # Anything else (punctuation, symbols, dates with separators, ...)
    return "generic"


def detect_dominant_format(
    series: pd.Series,
    sample_size: int = DOMINANT_FORMAT_SAMPLE_SIZE,
) -> dict:
    """
    Detect the dominant value-format of a column on a small head() sample.

    Returns:
      {
        "label":       "digit" | "alpha" | "alphanumeric" | "float" | "generic",
        "share_pct":   float,
        "sample_size": int     # how many non-missing rows were actually inspected
      }
    """
    # Take the first `sample_size` rows of the column, then drop missing values
    head_sample = series.head(sample_size)
    clean = _non_missing_series(head_sample)

    if clean.empty:
        return {
            "label": None,
            "share_pct": 0.0,
            "sample_size": 0,
        }

    labels = clean.map(_value_format_label)
    counts = labels.value_counts(dropna=False)
    total = int(counts.sum())

    return {
        "label": str(counts.index[0]),
        "share_pct": round(float(counts.iloc[0] / total * 100), 2),
        "sample_size": total,
    }


def collect_non_conforming_samples(
    series: pd.Series,
    dominant_label: str,
    sample_size: int = DOMINANT_FORMAT_SAMPLE_SIZE,
    max_samples: int = 100,
) -> dict:
    """
    Inspect the same head() sample used for dominant format detection
    and collect up to `max_samples` unique values whose format does NOT
    match the dominant label.

    Returns:
      {
        "share_pct": float,        # share of non-conforming values in the sample
        "samples":   list[str],    # up to `max_samples` unique non-conforming values
      }
    """
    head_sample = series.head(sample_size)
    clean = _non_missing_series(head_sample)

    if clean.empty or dominant_label is None:
        return {"share_pct": 0.0, "samples": []}

    labels = clean.map(_value_format_label)
    non_conforming_mask = labels != dominant_label
    non_conforming_values = clean.loc[non_conforming_mask]

    total = int(len(clean))
    n_non_conforming = int(non_conforming_mask.sum())

    unique_samples = []
    for v in non_conforming_values.tolist():
        native_v = _to_native(v)
        if native_v not in unique_samples:
            unique_samples.append(native_v)
        if len(unique_samples) >= max_samples:
            break

    return {
        "share_pct": round(float(n_non_conforming / total * 100), 2) if total else 0.0,
        "samples": unique_samples,
    }


def _try_numeric(series: pd.Series) -> pd.Series:
    """
    Convert a series to numeric where possible (supports comma decimals
    and optional thousands separators). Used only for the numeric_summary
    block in the rich profile.
    """
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")

    s = series.astype("string").str.strip()
    s = s.str.replace(".", "", regex=False)   # optional thousands separator
    s = s.str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")

In [34]:
# =============================================================================
# LLM-DRIVEN COLUMN ENRICHMENT
# =============================================================================
# Given the deterministic part of a column profile (stats + dominant format
# + non-conforming samples), call the LLM ONCE per column to obtain:
#   - role               : one of COLUMN_ROLES (or "custom:<name>")
#   - expected_dtype     : echoed from the role mapping (LLM may override)
#   - transformation_rule: declarative rule the cleaning agent will apply
#   - description        : 1-2 sentence human-readable description

# Closed set of declarative actions the LLM is allowed to choose from.
# The cleaning agent has ONE dispatcher per action; it does NOT need
# column-specific functions like `month_name_to_number`.
TRANSFORMATION_ACTIONS = {
    "none",               # all values already conform
    "extract_pattern",    # extract a substring matching target_format, drop the rest
    "strip_chars",        # strip given chars from start/end
    "pad_zeros",          # left-pad with zeros to a target length
    "normalize_case",     # upper/lower
    "map_to_canonical",   # for each non-conforming value, map to nearest canonical value
    "parse_date",         # parse a date according to a target format
    "coerce_numeric",     # coerce to number, NaN otherwise
    "set_null",           # explicitly null out broken placeholders
}


def _build_enrichment_prompt(deterministic_profile: dict) -> str:
    """Build the prompt sent to the LLM for a single column."""
    return f"""
You are a data profiling assistant.

You are given the deterministic profile of ONE column from a tabular dataset.
Your job is to enrich it with:
  1. a ROLE chosen from a closed vocabulary
  2. the EXPECTED pandas dtype the column should have AFTER cleaning
  3. a declarative TRANSFORMATION_RULE that explains how to convert
     non-conforming values into the dominant format
  4. a brief DESCRIPTION (1-2 sentences) for human reviewers

==============================================================================
CLOSED VOCABULARY OF ROLES
==============================================================================
{sorted(COLUMN_ROLES)}

If none fits, return a role of the form "custom:<short_snake_case_name>"
and explain why in the description.

==============================================================================
DEFAULT ROLE -> EXPECTED DTYPE MAPPING (for reference)
==============================================================================
{json.dumps(ROLE_TO_EXPECTED_DTYPE, indent=2)}

You MAY override the expected_dtype if you have a strong reason
(e.g. you assigned a custom role).

==============================================================================
CLOSED VOCABULARY OF TRANSFORMATION ACTIONS
==============================================================================
{sorted(TRANSFORMATION_ACTIONS)}

A transformation_rule MUST have the structure:
{{
  "action":        "<one of the actions above>",
  "target_format": "<short description of the expected value format>",
  "params":        {{...}},          // action-specific parameters, may be empty
  "description":   "<1 sentence explaining what the rule does>"
}}

Examples of params per action:
  - extract_pattern : {{"regex": "\\\\d+"}}                 // extract digits
  - strip_chars     : {{"chars": " \\t"}}                  // strip whitespace
  - pad_zeros       : {{"length": 2}}                      // pad to length 2
  - normalize_case  : {{"case": "lower"}}                  // or "upper"
  - map_to_canonical: {{"mapping": {{"<raw_value_1>": "<canonical_1>",
                                     "<raw_value_2>": "<canonical_2>"}},
                       "canonical_values": ["<canonical_1>", "<canonical_2>", "..."]}}
  - parse_date      : {{"input_formats": ["%d/%m/%Y","%Y-%m-%d"]}}
  - coerce_numeric  : {{"decimal_separator": ","}}         // optional
  - set_null        : {{}}                                  // no params
  - none            : {{}}                                  // no params

==============================================================================
NUMERIC SUMMARY ANALYSIS
==============================================================================
If the profile contains a "numeric_summary" field, you MUST use it to reason
about the quality of the column — not just the non_conforming samples.

The non_conforming samples only capture values whose CHARACTER FORMAT differs
from the dominant format. They cannot detect values that are numerically
implausible but correctly formatted (e.g. "24" looks like a valid digit just
like "2024", so it never appears in non_conforming).

When numeric_summary is present:
  - Compare min, max, mean and median to understand the expected value range.
  - If min or max are far from the median or from the expected domain, treat those outlier values
    as implausible and include a transformation rule that corrects them.
  - Choose the action that fixes ALL values universally, not just the ones
    you can enumerate. For example:
      * a year column where most values are 4-digit but some are 2-digit →
        use "extract_pattern" with a regex that extracts 4 digits, or
        "coerce_numeric" with a param that handles 2-digit-to-4-digit conversion
      * a count column where most values are 2-3 digits but a few are 0 or 1 →
        decide whether those are true zeros or missing data and act accordingly
  - NEVER use "map_to_canonical" to fix numerically implausible values:
    it only corrects the exact values you list, leaving all others untouched.

==============================================================================
COLUMN PROFILE
==============================================================================
{json.dumps(deterministic_profile, ensure_ascii=False, indent=2)}

==============================================================================
OUTPUT
==============================================================================
Return ONLY a JSON object with EXACTLY these keys, no preamble, no markdown:
{{
  "role":                "<role from the vocabulary or custom:...>",
  "expected_dtype":      "<pandas dtype string>",
  "transformation_rule": {{...}},
  "description":         "<1-2 sentences>"
}}
""".strip()


def _parse_llm_enrichment_response(response_text: str) -> dict:
    """Parse the LLM JSON response, tolerating ```json fences and extra whitespace."""
    text = response_text.strip()
    # Strip optional markdown fences
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return json.loads(text)


def _fallback_enrichment(deterministic_profile: dict, error: str) -> dict:
    """Used when the LLM call fails or returns invalid JSON."""
    return {
        "role": "unknown",
        "role_source": "fallback",
        "expected_dtype": "string",
        "transformation_rule": {
            "action": "none",
            "target_format": deterministic_profile.get("dominant_format", {}).get("label"),
            "params": {},
            "description": "Fallback: no transformation applied (LLM enrichment failed).",
        },
        "description": f"LLM enrichment failed: {error}",
    }


def _validate_and_fix_enrichment(
    enrichment: dict,
    deterministic_profile: dict,
) -> dict:
    """
    Apply post-validation rules to the LLM output to fix common incoherencies
    between role / expected_dtype / transformation_rule.
    Returns a corrected enrichment dict and a list of applied fixes.
    """
    fixes = []

    role = enrichment.get("role", "unknown")
    expected_dtype = enrichment.get("expected_dtype")
    rule = enrichment.get("transformation_rule") or {}
    action = rule.get("action", "none")
    params = rule.get("params") or {}

    dominant = (deterministic_profile.get("dominant_format") or {}).get("label")
    non_conforming_samples = (deterministic_profile.get("non_conforming") or {}).get("samples", [])

    # ── FIX 1: dtype coerente con il role ─────────────────────────────
    expected_dtype_from_role = resolve_expected_dtype(role)
    if role in ROLE_TO_EXPECTED_DTYPE and expected_dtype != expected_dtype_from_role:
        fixes.append(f"expected_dtype overridden from '{expected_dtype}' to '{expected_dtype_from_role}' to match role '{role}'")
        expected_dtype = expected_dtype_from_role

    # ── FIX 2: case 'title' / altri non supportati ────────────────────
    if action == "normalize_case":
        case = str(params.get("case", "lower")).lower()
        if case not in {"lower", "upper"}:
            fixes.append(f"normalize_case 'case={case}' is unsupported → forced to 'lower'")
            params = {**params, "case": "lower"}

    # ── FIX 3: digit + noise con coerce_numeric → switch a extract_pattern ─
    # Se il dominant è digit ma i non-conforming contengono digit + altro
    # (es. "1 voli", "~181"), coerce_numeric li butta a NaN.
    # extract_pattern li recupera estraendo le cifre.
    if dominant == "digit" and action == "coerce_numeric" and non_conforming_samples:
        has_digit_with_noise = any(
            (any(c.isdigit() for c in str(v)) and any(not c.isdigit() and c not in "-." for c in str(v).strip()))
            for v in non_conforming_samples
        )
        if has_digit_with_noise:
            fixes.append("dominant is 'digit' with noisy non-conforming values → action switched from 'coerce_numeric' to 'extract_pattern'")
            action = "extract_pattern"
            params = {"regex": r"-?\d+"}

    # ── FIX 4: map_to_canonical con valori stringa ma expected_dtype numerico ─
    # Se il role è numerico (month_number, year, ecc.) e la mappatura produce
    # stringhe come "01", "02", convertiamo i valori della mappatura a int.
    if action == "map_to_canonical" and expected_dtype in {"Int64", "Int32", "Float64", "Float32"}:
        mapping = params.get("mapping") or {}
        if mapping and any(isinstance(v, str) and v.lstrip("-").isdigit() for v in mapping.values()):
            new_mapping = {}
            for k, v in mapping.items():
                if isinstance(v, str) and v.lstrip("-").isdigit():
                    new_mapping[k] = int(v)
                else:
                    new_mapping[k] = v
            params = {**params, "mapping": new_mapping}
            fixes.append(f"map_to_canonical values converted from strings to integers to match expected_dtype '{expected_dtype}'")

    # ── FIX 5: textual format with high share but numeric role ────────
    # If the dominant_format is alpha/generic with share >= 80%, the
    # column is overwhelmingly textual and CANNOT be numeric, no matter
    # what the LLM inferred from the column name.
    NUMERIC_ROLES = {
        "identifier_numeric", "count", "measure", "percentage",
        "year", "month_number", "day_number",
    }
    dominant_share = (deterministic_profile.get("dominant_format") or {}).get("share_pct", 0)
    if dominant in {"alpha", "generic"} and dominant_share >= 80 and role in NUMERIC_ROLES:
        n_unique = deterministic_profile.get("n_unique_non_null", 0)
        row_count = deterministic_profile.get("row_count", 1)
        unique_ratio = n_unique / max(row_count, 1)

        # Heuristic: low cardinality => categorical, high cardinality => free text
        new_role = "code_categorical" if unique_ratio < 0.05 else "free_text"
        new_dtype = resolve_expected_dtype(new_role)
        new_action = "none"
        new_params = {}

        fixes.append(
            f"role '{role}' is numeric but dominant_format is '{dominant}' at "
            f"{dominant_share}% (textual evidence overrides column name). "
            f"Reassigned to '{new_role}', dtype to '{new_dtype}', action to '{new_action}'."
        )
        role = new_role
        expected_dtype = new_dtype
        action = new_action
        params = new_params

    # ── FIX 6: rule completa, sempre con tutti i campi ────────────────
    fixed_rule = {
        "action": action,
        "target_format": rule.get("target_format", dominant),
        "params": params,
        "description": rule.get("description", ""),
    }

    out = dict(enrichment)
    out["role"] = role
    out["expected_dtype"] = expected_dtype
    out["transformation_rule"] = fixed_rule
    if fixes:
        out["validator_fixes"] = fixes
    return out




def enrich_column_profile_with_llm(
    deterministic_profile: dict,
    llm_callable: Optional[Callable[[str], str]] = None,
) -> dict:
    """
    Call the LLM ONCE for a single column and return the enrichment dict
    with keys: role, role_source, expected_dtype, transformation_rule, description.
    """
    if llm_callable is None:
        return _fallback_enrichment(deterministic_profile, "no llm_callable provided")

    prompt = _build_enrichment_prompt(deterministic_profile)

    try:
        raw = llm_callable(prompt)
        parsed = _parse_llm_enrichment_response(str(raw))
    except Exception as e:
        return _fallback_enrichment(deterministic_profile, str(e))

    # Validate role
    role = parsed.get("role", "unknown")
    if role in COLUMN_ROLES:
        role_source = "llm"
    elif isinstance(role, str) and role.startswith("custom:"):
        role_source = "custom"
    else:
        role = "unknown"
        role_source = "fallback"

    # Resolve expected_dtype: prefer LLM value, fallback to deterministic mapping
    expected_dtype = parsed.get("expected_dtype") or resolve_expected_dtype(role)

    # Validate transformation_rule
    rule = parsed.get("transformation_rule") or {}
    if not isinstance(rule, dict) or rule.get("action") not in TRANSFORMATION_ACTIONS:
        rule = {
            "action": "none",
            "target_format": deterministic_profile.get("dominant_format", {}).get("label"),
            "params": {},
            "description": "LLM returned an invalid transformation_rule; defaulted to 'none'.",
        }
    rule.setdefault("params", {})
    rule.setdefault("target_format", deterministic_profile.get("dominant_format", {}).get("label"))
    rule.setdefault("description", "")

    enrichment = {
        "role": role,
        "role_source": role_source,
        "expected_dtype": expected_dtype,
        "transformation_rule": rule,
        "description": str(parsed.get("description", "")).strip(),
    }

    # Post-validate and fix incoherencies
    enrichment = _validate_and_fix_enrichment(enrichment, deterministic_profile)

    return enrichment

In [35]:
# =============================================================================
# CLEANING HELPERS — generic, hard-coded-free
# =============================================================================
# Reusable building blocks for the cleaning agent. They are completely
# generic: no column-specific knowledge, no domain dictionaries, no
# regex tied to a specific data source.

import re
import json
import unicodedata
from typing import Optional, Iterable


DEFAULT_CLEANING_MISSING_TOKENS = {
    "", " ", "n/a", "na", "null", "none", "unknown", "nan",
    "-", "--", "_", "nd", "n.d.", "n/d", "missing"
}


def _strip_accents(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch))


def _collapse_spaces(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


# ── Column name normalization ─────────────────────────────────────────

def to_snake_case(name: str) -> str:
    """
    Convert a column name to lowercase snake_case in a stable way.
    Handles spaces, punctuation, camelCase, PascalCase and noisy symbols.
    """
    name = "" if name is None else str(name)
    name = _strip_accents(name).strip()

    # Separate camelCase / PascalCase boundaries
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    name = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1_\2", name)

    # Replace non-alphanumeric runs with underscore
    name = re.sub(r"[^0-9A-Za-z]+", "_", name)

    # Collapse underscores and trim
    name = re.sub(r"_+", "_", name).strip("_").lower()

    if not name:
        name = "unnamed_column"

    # If the name starts with a digit, prefix it to keep it identifier-like
    if re.match(r"^\d", name):
        name = f"col_{name}"

    return name


def normalize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with column names normalized to snake_case."""
    out = df.copy()
    out.columns = [to_snake_case(col) for col in out.columns]
    return out


def deduplicate_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Append __1, __2, ... suffixes to columns whose name appears more than once."""
    out = df.copy()
    seen = {}
    new_cols = []

    for col in out.columns:
        if col not in seen:
            seen[col] = 0
            new_cols.append(col)
        else:
            seen[col] += 1
            new_cols.append(f"{col}__{seen[col]}")

    out.columns = new_cols
    return out


# ── Row / value normalization ─────────────────────────────────────────

def remove_duplicate_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Drop exact duplicate rows."""
    return df.drop_duplicates().copy()


def standardize_missing_values(
    df: pd.DataFrame,
    missing_tokens: Optional[Iterable[str]] = None,
) -> pd.DataFrame:
    """
    Convert semantic missing placeholders (e.g. "n/a", "-", "unknown") to pd.NA.
    Only applied to string-like values; native NaN and non-strings are untouched.
    """
    out = df.copy()
    tokens = {str(x).strip().lower() for x in (missing_tokens or DEFAULT_CLEANING_MISSING_TOKENS)}

    for col in out.columns:
        out[col] = out[col].apply(
            lambda x: pd.NA
            if isinstance(x, str) and x.strip().lower() in tokens
            else x
        )
    return out


def normalize_text_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Lowercase and whitespace-normalize textual cells only.
    Keeps nulls and non-text cells unchanged.
    """
    out = df.copy()

    def _norm(x):
        if pd.isna(x):
            return x
        if isinstance(x, str):
            return _collapse_spaces(x).lower()
        return x

    for col in out.columns:
        out[col] = out[col].apply(_norm)

    return out


# ── Cleaning plan loader (reads the SLIM json produced by profiling) ──

def load_cleaning_plan(plan_json_path: str, dataset_key: str) -> dict:
    """
    Load the slim cleaning plan for a single dataset.

    The slim JSON has structure:
      { dataset_key: { col_name: {expected_dtype, dominant_format, transformation_rule} } }

    Returns the per-column cleaning plan for `dataset_key`, or {} if missing.
    """
    with open(plan_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data.get(dataset_key, {})

In [36]:
# =============================================================================
# COLUMN PROFILING — orchestration
# =============================================================================
# Builds the rich per-column profile (deterministic stats + LLM enrichment)
# and then derives the slim cleaning plan that will be fed to the cleaning agent.

def profile_single_column(
    df: pd.DataFrame,
    column_name: str,
    llm_callable: Optional[Callable[[str], str]] = None,
    sample_size: int = DOMINANT_FORMAT_SAMPLE_SIZE,
) -> dict:
    """
    Build the full RICH profile for one column:
      1. deterministic stats (pandas-based)
      2. dominant format on a head() sample
      3. non-conforming samples on the same head() sample
      4. numeric_summary (only if the column is mostly numeric)
      5. LLM enrichment: role, role_source, expected_dtype, transformation_rule, description
    """
    series = df[column_name]
    missing_mask = _normalize_missing_mask(series)
    non_missing = series.loc[~missing_mask]

    # ── Deterministic stats ────────────────────────────────────────────
    sample_values = [_to_native(v) for v in non_missing.head(5).tolist()]

    dominant = detect_dominant_format(series, sample_size=sample_size)
    non_conforming = collect_non_conforming_samples(
        series,
        dominant_label=dominant["label"],
        sample_size=sample_size,
        max_samples=100,
    )

    deterministic_profile = {
        "column_name": column_name,
        "dtype": str(series.dtype),
        "row_count": int(len(series)),
        "null_count": int(missing_mask.sum()),
        "non_null_count": int((~missing_mask).sum()),
        "missing_percentage": round(float(missing_mask.mean() * 100), 2),
        "n_unique_non_null": int(non_missing.nunique(dropna=True)),
        "unique_ratio_non_null": round(
            float(non_missing.nunique(dropna=True) / max(len(non_missing), 1)), 4
        ),
        "sample_values": sample_values,
        "dominant_format": dominant,
        "non_conforming": non_conforming,
    }

    # ── Numeric summary (only if mostly parseable as numeric) ──────────
    numeric = _try_numeric(non_missing)
    if len(non_missing) > 0 and numeric.notna().mean() >= 0.9:
        numeric = numeric.dropna()
        if not numeric.empty:
            deterministic_profile["numeric_summary"] = {
                "min": _to_native(numeric.min()),
                "max": _to_native(numeric.max()),
                "mean": _to_native(round(float(numeric.mean()), 4)),
                "median": _to_native(round(float(numeric.median()), 4)),
                "std": _to_native(round(float(numeric.std(ddof=1)), 4)) if len(numeric) > 1 else None,
            }

    # ── LLM enrichment (role + transformation_rule + description) ──────
    enrichment = enrich_column_profile_with_llm(
        deterministic_profile=deterministic_profile,
        llm_callable=llm_callable,
    )

    full_profile = {**deterministic_profile, **enrichment}
    return _to_native(full_profile)


def profile_all_columns(
    df: pd.DataFrame,
    llm_callable: Optional[Callable[[str], str]] = None,
    sample_size: int = DOMINANT_FORMAT_SAMPLE_SIZE,
) -> dict:
    """Build the rich profile for every column of a dataframe."""
    return {
        col: profile_single_column(df, col, llm_callable=llm_callable, sample_size=sample_size)
        for col in df.columns
    }


# =============================================================================
# LLM description (kept for human debugging — optional)
# =============================================================================

def generate_column_description_with_llm(
    column_profile: dict,
    llm_callable: Optional[Callable[[str], str]] = None,
) -> str:
    """
    Returns the description already produced by the LLM enrichment step.
    Kept as a convenience accessor so external code can still ask for the
    human-friendly description of a column profile.
    """
    description = column_profile.get("description")
    if description:
        return str(description).strip()

    # Fallback if for any reason the enrichment didn't include a description.
    dominant = column_profile.get("dominant_format", {}) or {}
    return (
        f"Column '{column_profile.get('column_name')}' has dtype "
        f"{column_profile.get('dtype')}, dominant format "
        f"{dominant.get('label')} ({dominant.get('share_pct')}%), "
        f"and {column_profile.get('missing_percentage')}% missing values."
    )


# =============================================================================
# Slim cleaning plan
# =============================================================================
# Distilled view of the rich profile. Contains ONLY what the cleaning agent
# needs to operate. Fed to the cleaning step as cleaning_plan.json.

def build_cleaning_plan(rich_profile_per_column: dict) -> dict:
    """
    Reduce the rich per-column profile into the slim cleaning plan.
    Input  : {col_name: rich_profile_dict, ...}
    Output : {col_name: {expected_dtype, dominant_format, transformation_rule}}
    """
    plan = {}
    for col_name, profile in rich_profile_per_column.items():
        dominant = profile.get("dominant_format", {}) or {}
        plan[col_name] = {
            "column_name": col_name,
            "expected_dtype": profile.get("expected_dtype", "string"),
            "dominant_format": dominant.get("label"),
            "transformation_rule": profile.get("transformation_rule", {
                "action": "none",
                "target_format": dominant.get("label"),
                "params": {},
                "description": "",
            }),
        }
    return plan


# =============================================================================
# Main orchestration — writes BOTH JSON files
# =============================================================================

def build_dataset_column_profiles(
    df: pd.DataFrame,
    dataset_name: str,
    rich_json_path: str,
    slim_json_path: str,
    llm_callable: Optional[Callable[[str], str]] = None,
    sample_size: int = DOMINANT_FORMAT_SAMPLE_SIZE,
    findings_root_key: str = "column_profiles",
) -> dict:
    """
    Build the rich JSON profile for every column and ALSO write the slim
    cleaning plan derived from it.

    Rich JSON structure:
      {
        findings_root_key: {
          dataset_name: {
            "dataset_summary": {...},
            "columns": {col_name: rich_profile, ...}
          }
        }
      }

    Slim JSON structure:
      {
        dataset_name: {col_name: {expected_dtype, dominant_format, transformation_rule}}
      }
    """
    # ── Build rich profile ─────────────────────────────────────────────
    findings = _safe_load_json(rich_json_path)

    dataset_summary = {
        "dataset_name": dataset_name,
        "shape": get_dataframe_shape(df),
        "dtypes": get_dataframe_dtypes(df),
    }

    columns_payload = profile_all_columns(
        df,
        llm_callable=llm_callable,
        sample_size=sample_size,
    )

    findings.setdefault(findings_root_key, {})
    findings[findings_root_key][dataset_name] = {
        "dataset_summary": dataset_summary,
        "columns": columns_payload,
    }
    _safe_write_json(rich_json_path, findings)

    # ── Build slim cleaning plan ───────────────────────────────────────
    slim_all = _safe_load_json(slim_json_path)
    slim_all[dataset_name] = build_cleaning_plan(columns_payload)
    _safe_write_json(slim_json_path, slim_all)

    return findings[findings_root_key][dataset_name]


def build_dataset_column_profiles_from_csv(
    csv_path: str,
    rich_json_path: str,
    slim_json_path: str,
    llm_callable: Optional[Callable[[str], str]] = None,
    dataset_name: Optional[str] = None,
    sample_size: int = 200,
    read_csv_kwargs: Optional[dict] = None,
) -> dict:
    """
    Convenience wrapper that loads a CSV and runs the full profiling pipeline.

    Column names are normalized to snake_case and deduplicated BEFORE profiling,
    so that the keys produced in the cleaning plan match the keys the cleaning
    agent will see after applying the same normalization in its own phase 1.
    """
    read_csv_kwargs = read_csv_kwargs or {}
    dataset_name = dataset_name or os.path.splitext(os.path.basename(csv_path))[0]

    df = pd.read_csv(csv_path, **read_csv_kwargs)

    # Apply the SAME column-name normalization the cleaning agent will apply.
    df = normalize_column_names(df)
    df = deduplicate_column_names(df)

    return build_dataset_column_profiles(
        df=df,
        dataset_name=dataset_name,
        rich_json_path=rich_json_path,
        slim_json_path=slim_json_path,
        llm_callable=llm_callable,
        sample_size=sample_size,
    )

In [37]:
# =============================================================================
# TRANSFORMATION DISPATCHER
# =============================================================================
# One vectorized function per action listed in TRANSFORMATION_ACTIONS.
# Each function takes a pandas Series and the rule's `params` dict, and
# returns a transformed Series. The functions are deliberately generic:
# no column-specific knowledge.
#
# IMPORTANT: each transformation is applied ONLY to rows whose current
# value does NOT already match the dominant_format. Rows already
# conforming are left untouched.

# ── Format conformity check (mirrors the profiling label) ─────────────

def _row_matches_format(series: pd.Series, dominant_label: Optional[str]) -> pd.Series:
    """
    Return a boolean mask: True where the value already matches the
    dominant format label produced during profiling.
    """
    if dominant_label is None:
        return pd.Series(False, index=series.index)
    return series.map(lambda v: _value_format_label(v) == dominant_label)


# ── Per-action transformations ────────────────────────────────────────

def _action_none(series: pd.Series, params: dict) -> pd.Series:
    return series


def _action_extract_pattern(series: pd.Series, params: dict) -> pd.Series:
    pattern = params.get("regex", r"\d+")
    extracted = series.astype("string").str.extract(f"({pattern})", expand=False)
    return extracted


def _action_strip_chars(series: pd.Series, params: dict) -> pd.Series:
    chars = params.get("chars", None)   # None => strip whitespace
    return series.astype("string").str.strip(chars)


def _action_pad_zeros(series: pd.Series, params: dict) -> pd.Series:
    length = int(params.get("length", 2))
    return series.astype("string").str.zfill(length)


def _action_normalize_case(series: pd.Series, params: dict) -> pd.Series:
    case = params.get("case", "lower")
    s = series.astype("string")
    if case == "upper":
        return s.str.upper()
    return s.str.lower()


def _action_map_to_canonical(series: pd.Series, params: dict) -> pd.Series:
    """
    Replace each value with its mapped canonical value.
    `mapping` is a dict {raw_value_lowercased: canonical_value}.
    Values not in the mapping are left unchanged (not nulled).

    Robustness: the mapping value may come from the profiling LLM as
    non-string types (int, float, bool, None). We coerce non-null values
    to string so the assignment is compatible with pandas StringDtype
    columns; the final cast to the expected dtype is performed later
    by enforce_expected_dtype. A None mapping value is interpreted as
    "null this cell" and returns pd.NA.
    """
    mapping = params.get("mapping", {}) or {}
    if not mapping:
        return series

    # Lowercase keys for case-insensitive matching; normalize values.
    def _norm_value(v):
        if v is None:
            return pd.NA
        return str(v)

    norm_mapping = {
        str(k).strip().lower(): _norm_value(v)
        for k, v in mapping.items()
    }

    def _map_one(v):
        if pd.isna(v):
            return v
        key = str(v).strip().lower()
        return norm_mapping.get(key, v)

    return series.map(_map_one)


def _action_parse_date(series: pd.Series, params: dict) -> pd.Series:
    """
    Try parsing dates with the given input formats in order; the first one
    that yields a valid date wins. Falls back to pandas auto-parsing.
    """
    input_formats = params.get("input_formats", []) or []

    if not input_formats:
        return pd.to_datetime(series, errors="coerce")

    parsed = pd.Series(pd.NaT, index=series.index)
    remaining_mask = series.notna()

    for fmt in input_formats:
        attempt = pd.to_datetime(series[remaining_mask], format=fmt, errors="coerce")
        # Place successful parses into `parsed`
        success_idx = attempt.dropna().index
        parsed.loc[success_idx] = attempt.loc[success_idx]
        # Update remaining mask: still-unparsed rows
        remaining_mask = parsed.isna() & series.notna()
        if not remaining_mask.any():
            break

    # Last resort: auto-parse anything still missing
    if remaining_mask.any():
        parsed.loc[remaining_mask] = pd.to_datetime(
            series.loc[remaining_mask], errors="coerce"
        )

    return parsed


def _action_coerce_numeric(series: pd.Series, params: dict) -> pd.Series:
    """
    Coerce to numeric. Supports a custom decimal_separator (e.g. ',' for italian).
    """
    decimal_sep = params.get("decimal_separator", ".")
    s = series.astype("string").str.strip()

    if decimal_sep == ",":
        # Remove dot thousands separator, then convert comma to dot
        s = s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)

    return pd.to_numeric(s, errors="coerce")


def _action_set_null(series: pd.Series, params: dict) -> pd.Series:
    return pd.Series(pd.NA, index=series.index)


# ── Action registry ───────────────────────────────────────────────────

ACTION_DISPATCHER = {
    "none":              _action_none,
    "extract_pattern":   _action_extract_pattern,
    "strip_chars":       _action_strip_chars,
    "pad_zeros":         _action_pad_zeros,
    "normalize_case":    _action_normalize_case,
    "map_to_canonical":  _action_map_to_canonical,
    "parse_date":        _action_parse_date,
    "coerce_numeric":    _action_coerce_numeric,
    "set_null":          _action_set_null,
}


def apply_transformation_rule(
    series: pd.Series,
    transformation_rule: dict,
    dominant_label: Optional[str],
) -> pd.Series:
    """
    Apply the LLM-generated transformation_rule to a column, ONLY on rows
    that do not already match the dominant format. Conforming rows are
    left untouched.

    The output series is coerced to `object` dtype before the in-place
    assignment, so that action functions are free to return whatever
    dtype is most natural for them (string, int, float, Timestamp, NA).
    The final dtype cast is the sole responsibility of
    enforce_expected_dtype, called separately by the caller.
    """
    action = (transformation_rule or {}).get("action", "none")
    params = (transformation_rule or {}).get("params", {}) or {}

    if action == "none" or action not in ACTION_DISPATCHER:
        return series

    # Identify rows that need the transformation
    conforming_mask = _row_matches_format(series, dominant_label)
    target_mask = ~conforming_mask & series.notna()

    if not target_mask.any():
        return series

    # Coerce to object dtype so the assignment can accept ANY return type
    # from the action dispatcher (string, int, float, Timestamp, pd.NA).
    # The final cast to expected_dtype is done later by enforce_expected_dtype.
    out = series.astype("object").copy()

    fn = ACTION_DISPATCHER[action]
    out.loc[target_mask] = fn(out.loc[target_mask], params)
    return out


# ── Final dtype enforcement ───────────────────────────────────────────

def enforce_expected_dtype(series: pd.Series, expected_dtype: str) -> pd.Series:
    """
    Convert the series to the expected pandas dtype declared by the profile.
    Falls back to leaving the series untouched if the cast cannot be done.
    """
    if expected_dtype is None or str(series.dtype) == expected_dtype:
        return series

    try:
        if expected_dtype.startswith("datetime"):
            return pd.to_datetime(series, errors="coerce")
        if expected_dtype in {"Int64", "Int32"}:
            return pd.to_numeric(series, errors="coerce").astype(expected_dtype)
        if expected_dtype in {"Float64", "Float32", "float64", "float32"}:
            return pd.to_numeric(series, errors="coerce").astype(expected_dtype)
        return series.astype(expected_dtype)
    except Exception:
        return series


# =============================================================================
# CLEANING ORCHESTRATOR
# =============================================================================
# Two-phase deterministic cleaning:
#   PHASE 1 (global pandas-base steps): snake_case columns, deduplicate
#           column names, standardize missing tokens, lowercase text values.
#   PHASE 2 (per-column LLM-driven transformations): for each column listed
#           in the cleaning plan, apply the transformation_rule on
#           non-conforming rows, then enforce the expected_dtype.
#   PHASE 3 (final global step): drop exact duplicate rows.

def clean_dataset(
    input_csv_path: str,
    output_csv_path: str,
    cleaning_plan_path: str,
    dataset_key: str,
    read_csv_kwargs: Optional[dict] = None,
) -> dict:
    """
    Run the full deterministic cleaning pipeline on a single dataset.
    Returns a small report dict (shapes, columns affected, dtype changes).
    """
    read_csv_kwargs = read_csv_kwargs or {}

    # Load
    df = pd.read_csv(input_csv_path, **read_csv_kwargs)
    shape_before = df.shape

    # Load slim cleaning plan
    plan = load_cleaning_plan(cleaning_plan_path, dataset_key)

    # ── PHASE 1: global pandas-base steps ────────────────────────────
    df = normalize_column_names(df)
    df = deduplicate_column_names(df)
    df = standardize_missing_values(df)
    df = normalize_text_values(df)

    # ── PHASE 2: per-column LLM-driven cleaning ──────────────────────
    per_column_actions = {}
    dtype_changes = {}

    for col in df.columns:
        col_plan = plan.get(col)
        if not col_plan:
            per_column_actions[col] = {"action": "skipped (no plan)"}
            continue

        rule = col_plan.get("transformation_rule", {}) or {}
        dominant_label = col_plan.get("dominant_format")
        expected_dtype = col_plan.get("expected_dtype")

        # Apply the transformation only on non-conforming rows
        original = df[col]
        df[col] = apply_transformation_rule(
            series=df[col],
            transformation_rule=rule,
            dominant_label=dominant_label,
        )

        # Enforce final dtype
        dtype_before = str(df[col].dtype)
        df[col] = enforce_expected_dtype(df[col], expected_dtype)
        dtype_after = str(df[col].dtype)

        n_changed = int((original.astype("string") != df[col].astype("string")).sum())
        per_column_actions[col] = {
            "action": rule.get("action", "none"),
            "rows_changed": n_changed,
            "dtype_before": dtype_before,
            "dtype_after": dtype_after,
        }
        if dtype_before != dtype_after:
            dtype_changes[col] = {"before": dtype_before, "after": dtype_after}

    # ── PHASE 3: final global step ───────────────────────────────────
    df = remove_duplicate_rows(df)
    shape_after = df.shape

    # Save
    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
    df.to_csv(output_csv_path, index=False)

    return {
        "dataset_key": dataset_key,
        "input_path": input_csv_path,
        "output_path": output_csv_path,
        "shape_before": list(shape_before),
        "shape_after": list(shape_after),
        "duplicate_rows_removed": int(shape_before[0] - shape_after[0]),
        "per_column_actions": per_column_actions,
        "dtype_changes": dtype_changes,
    }

# Tool REPL

In [38]:
# EXECUTOR_GLOBALS = {
#     "pd": pd,
#     "np": np,
#     "json": json,
#     "re": re,
#     "os": os,

#     # cleaning helpers
#     "to_snake_case": to_snake_case,
#     "normalize_column_names": normalize_column_names,
#     "deduplicate_column_names": deduplicate_column_names,
#     "standardize_missing_values": standardize_missing_values,
#     "normalize_text_values": normalize_text_values,
#     "remove_duplicate_rows": remove_duplicate_rows,
#     "month_name_to_number": month_name_to_number,
#     "parse_italian_float": parse_italian_float,

#     # profile helpers
#     "load_dataset_profile": load_dataset_profile,
#     "get_profile_missing_tokens": get_profile_missing_tokens,
#     "get_profile_semantic_map": get_profile_semantic_map,
# }


# class PythonREPL:
#     """Executes Python code and captures output."""
#     def __init__(self, initial_globals=None):
#         self.globals = dict(initial_globals or {})

#     def run(self, code: str) -> str:
#         old_stdout, old_stderr = sys.stdout, sys.stderr
#         sys.stdout = mystdout = io.StringIO()
#         sys.stderr = mystderr = io.StringIO()
#         try:
#             exec(code, self.globals)
#             output = mystdout.getvalue()
#             errors = mystderr.getvalue()
#             if errors:
#                 output += f"\nSTDERR:\n{errors}"
#             return output if output else "(OK, no output)"
#         except Exception:
#             return f"Error:\n{traceback.format_exc()}"
#         finally:
#             sys.stdout, sys.stderr = old_stdout, old_stderr


# _repl = PythonREPL(initial_globals=EXECUTOR_GLOBALS)

# Prompts
Natural language prompts for each pipeline task.
The LLM receives these and generates Python code autonomously.

In [39]:
# # ── Findings helper ──────────────────────────────────────────────────────────

# def _findings_guidance(task_key: str, extra_notes: str = "") -> str:
#     base = (
#         f"Maintain a shared findings JSON at '{FINDINGS_JSON}'. "
#         f"At the start, attempt to load it; if missing, empty, invalid, or corrupted, initialize an empty dict instead of failing. "
#         f"Store new information under the key '{task_key}' while preserving existing keys for other tasks. "
#         f"Use concise, machine-readable fields with only native Python JSON-serializable types such as dict, list, str, int, float, bool, or null. "
#         f"Convert pandas and numpy values to native Python types before saving. "
#         f"Convert tuples to lists before saving. "
#         # f"After completing the task, update the entry for '{task_key}' and write the full JSON back by overwriting the file. "
#     )
#     if extra_notes:
#         base += extra_notes
#     return base


# # ── Task 1: Clean the dataset ─────────────────────────────────────────────────

# #def _build_structural_cleaning_prompt(input_path, output_path, profile_json_path, dataset_profile_key):
#     return (
#         f"You are a data cleaning agent for tabular datasets used in downstream merge and anomaly detection tasks. "
#         f"Load the dataset at '{input_path}' and produce a cleaned version saved to '{output_path}'. "
#         f"Before cleaning, load the dataset profile from '{profile_json_path}' using dataset key '{dataset_profile_key}'. "

#         f"The execution environment already provides these callable helper functions: "
#         f"to_snake_case, normalize_column_names, deduplicate_column_names, "
#         f"standardize_missing_values, normalize_text_values, remove_duplicate_rows, "
#         f"load_dataset_profile, get_profile_missing_tokens, get_profile_semantic_map. "

#         f"Do not redefine these helpers. "
#         f"Do not create placeholder implementations. "
#         f"Do not create no-op stubs. "
#         f"If any helper is unavailable, fail explicitly with a clear error. "

#         f"Execution plan: "
#         f"1) load CSV, "
#         f"2) capture original shape and original columns, "
#         f"3) load matching dataset profile, "
#         f"4) compute missing tokens from profile, "
#         f"5) normalize column names with normalize_column_names(df), "
#         f"6) deduplicate column names with deduplicate_column_names(df), "
#         f"7) standardize semantic missing values, "
#         f"8) normalize textual values to lowercase, "
#         f"9) remove exact duplicate rows, "
#         f"10) validate and save. "

#         f"Validation requirements are mandatory: "
#         f"all final column names must be lowercase snake_case, "
#         f"there must be no duplicate column names, "
#         f"and textual values must be lowercase. "

#         f"After normalize_column_names(df), explicitly verify that every column name equals to_snake_case(column_name). "
#         f"If any final column still contains uppercase letters, spaces, or symbols such as percent signs, fail explicitly. "

#         f"After normalize_text_values(df), explicitly verify on a sample of textual cells that uppercase text was normalized. "

#         f"Use the dataset profile as structural guidance, but preserve information conservatively. "
#         f"Do not aggressively drop sparse columns. "

#         f"Save the cleaned dataframe to '{output_path}' without index. "
#         f"Print only a short summary with original shape, final shape, final columns, duplicate rows removed, and validation status. "
#     )

# def _build_structural_cleaning_prompt(
#     input_path,
#     output_path,
#     profile_json_path,
#     dataset_profile_key,
#     findings_key,
# ):
#     return (
#         # ── Role & constraints ────────────────────────────────────────────
#         f"You are a profile-driven data cleaning agent for tabular datasets "
#         f"used in downstream merge and anomaly detection tasks. "
#         f"Write Python code only. "
#         f"Do not define functions. "
#         f"Do not import local modules. "
#         f"Use only pandas and the provided helper functions. "

#         # ── Available helpers (toolkit) ───────────────────────────────────
#         f"The following helper functions are already available in the execution environment: "
#         f"to_snake_case, normalize_column_names, deduplicate_column_names, "
#         f"standardize_missing_values, normalize_text_values, remove_duplicate_rows, "
#         f"month_name_to_number, parse_italian_float, "
#         f"load_dataset_profile, get_profile_missing_tokens, get_profile_semantic_map. "
#         f"Do not redefine these helpers. "
#         f"Always load the profile using load_dataset_profile; do not open the JSON file manually. "

#         # ── Profile structure (CRITICAL) ──────────────────────────────────
#         f"The profile returned by load_dataset_profile is a dict with two top-level keys: "
#         f"'dataset_summary' and 'columns'. "
#         f"Per-column metadata lives in profile['columns'][<column_name>]. "
#         f"Each column entry contains these fields: "
#         f"column_name (str), dtype (str), row_count (int), null_count (int), "
#         f"non_null_count (int), missing_percentage (float in 0-100), "
#         f"n_unique_non_null (int), unique_ratio_non_null (float in 0-1), "
#         f"sample_values (list), dominant_format (nested dict with keys "
#         f"'dominant_format', 'dominant_format_share' in 0-100, and 'minority_formats'), "
#         f"semantic_inference (nested dict with 'semantic_family' in {{'numeric','string'}}). "
#         f"Never assume a column entry exists at profile[col]; always use profile['columns'][col]. "
#         f"dominant_format is a dict, so read "
#         f"profile['columns'][col]['dominant_format']['dominant_format'] "
#         f"and profile['columns'][col]['dominant_format']['dominant_format_share']. "
#         f"minority_formats is a list of dicts; each dict has keys: "
#         f"format_label (str), count (int), share (float in 0-100), "
#         f"sample_values (list), and suggested_transformation (str). "
#         f"The possible values of suggested_transformation are: "
#         f"'strip_whitespace', 'month_name_to_number', 'parse_italian_float', "
#         f"'extract_digits', 'parse_float', 'preserve'. "
#         f"semantic_family is inside "
#         f"profile['columns'][col]['semantic_inference']['semantic_family']. "

#         # ── Fixed initial steps (NOT profile-driven) ──────────────────────
#         f"Execute these fixed initial steps in order: "
#         f"1) Load the CSV file at '{input_path}' into a pandas DataFrame called df. "
#         f"2) Capture shape_before and original column names for findings. "
#         f"3) Load the profile using load_dataset_profile('{profile_json_path}', '{dataset_profile_key}'). "
#         f"4) Apply df = normalize_column_names(df). "
#         f"5) Apply df = deduplicate_column_names(df). "
#         f"6) Extract missing tokens with get_profile_missing_tokens(profile) "
#         f"and apply df = standardize_missing_values(df, missing_tokens=<those_tokens>). "
#         f"7) Apply df = normalize_text_values(df) to lowercase and whitespace-normalize all textual values. "

#         # ── Profile-driven per-column cleaning ────────────────────────────
#         f"Then perform profile-driven per-column cleaning. "
#         f"Iterate over df.columns. For each column, look it up in profile['columns']. "
#         f"If the column is not present in the profile, skip it and record the decision. "
#         f"You may apply zero, one, or more of the following allowed actions per column. "
#         f"Do not invent new actions. "

#         f"Allowed actions: "
#         f"(1) drop_column: use when the column is structurally unusable, typically when "
#         f"missing_percentage is very high or n_unique_non_null equals 1. "
#         f"(2) drop_rows: use when a small number of rows are clearly problematic and "
#         f"missing_percentage is very low. "
#         f"(3) cast_dtype: use when dtype is 'str' but semantic_inference.semantic_family is 'numeric', "
#         f"or when dominant_format.dominant_format is 'numeric_string'. "
#         f"Apply with df[col] = pd.to_numeric(df[col], errors='coerce'). "
#         f"IMPORTANT: cast_dtype must be applied AFTER apply_minority_transformations below, "
#         f"so that minority values already fixed contribute as valid numerics. "
#         f"(4) impute_missing: use when missing_percentage is low. "
#         f"For semantic_family 'numeric' use median; for 'string' use mode. "
#         f"(5) collapse_rare_categories: use only when semantic_family is 'string', "
#         f"n_unique_non_null is small, and sample_values clearly shows equivalent variants "
#         f"not already unified by the global lowercase normalization. "
#         f"(6) apply_minority_transformations: iterate over "
#         f"profile['columns'][col]['dominant_format']['minority_formats']. "
#         f"For each minority entry, read its suggested_transformation tag and its sample_values. "
#         f"Build a boolean mask that identifies the rows of df[col] whose current value "
#         f"(compared as stripped lowercase string) equals any of the sample_values of that minority entry. "
#         f"Apply the transformation ONLY to the masked rows, not to the entire column: "
#         f"  - 'strip_whitespace': df.loc[mask, col] = df.loc[mask, col].astype(str).str.strip(); "
#         f"  - 'month_name_to_number': df.loc[mask, col] = df.loc[mask, col].apply(month_name_to_number); "
#         f"  - 'parse_italian_float': df.loc[mask, col] = df.loc[mask, col].apply(parse_italian_float); "
#         f"  - 'extract_digits': df.loc[mask, col] = df.loc[mask, col].astype(str).str.extract(r'(-?\\d+)', expand=False); "
#         f"  - 'parse_float': df.loc[mask, col] = pd.to_numeric(df.loc[mask, col], errors='coerce'); "
#         f"  - 'preserve': do nothing for that minority entry. "
#         f"Do not apply a transformation to values outside the minority sample set. "

#         # ── Decision policy ───────────────────────────────────────────────
#         f"Use the profile as the sole source of truth. "
#         f"Evaluate each allowed action independently per column; do not chain them with elif, "
#         f"so a single column may receive multiple actions in the same run "
#         f"(for example apply_minority_transformations followed by cast_dtype). "
#         f"Be conservative: when in doubt, do not act. "
#         f"Prefer preserving information over aggressive cleaning. "
#         f"Do not collapse distinct categories. "
#         f"Do not change the semantic meaning of a column. "

#         # ── Fixed final steps ─────────────────────────────────────────────
#         f"After per-column cleaning, apply df = remove_duplicate_rows(df) as a final fixed step. "

#         # ── Validation & save ─────────────────────────────────────────────
#         f"Before saving, validate that df is non-empty and has unique column names. "
#         f"Do not save any output if those checks fail. "
#         f"Save the cleaned dataframe to '{output_path}' without index. "
#         f"Print a short summary with shape_before, shape_after, final column names, "
#         f"columns dropped, rows dropped, and duplicate rows removed. "

#         # ── Findings guidance ─────────────────────────────────────────────
#         + _findings_guidance(
#             findings_key,
#             "Capture shape_before, shape_after, columns_final (list), "
#             "duplicate_rows_removed (int), and per_column_decisions (list). "
#             "Each entry of per_column_decisions must be a dict with: "
#             "column_name (str), "
#             "actions_taken (list of strings from "
#             "{'drop_column','drop_rows','cast_dtype','impute_missing',"
#             "'collapse_rare_categories','apply_minority_transformations'}), "
#             "reason (str, citing the exact profile field and value), "
#             "action_details (dict, e.g. rows_dropped_count, new_dtype, imputed_value, "
#             "collapsed_mapping, minority_transformations_applied with a list of "
#             "{format_label, suggested_transformation, rows_affected}). "
#             "Columns with no action must still appear with actions_taken=[] "
#             "and reason='no profile signal for action'. "
#         )
#     )


# def _build_data_prompt_1():
#     return _build_structural_cleaning_prompt(
#         ALLARMI_CSV,
#         f"{OUTPUT_DIR}/allarmi_clean.csv",
#         COLUMN_PROFILES_JSON,
#         "allarmi_raw",
#         "data_loading_allarmi",
#     )

# def _build_data_prompt_2():
#     return _build_structural_cleaning_prompt(
#         TIPOLOGIA_CSV,
#         f"{OUTPUT_DIR}/tipologia_clean.csv",
#         COLUMN_PROFILES_JSON,
#         "tipologia_raw",
#         "data_loading_tipologia",
#     )


# # ── Task 2: Normalize the dataset ─────────────────────────────────────────────

# def _build_semantic_normalization_prompt(input_path, output_path, findings_key):
#     return (
#         f"You are a semantic normalization agent for tabular datasets already cleaned at schema level. "
#         f"Load the dataset at '{input_path}' and produce a semantically refined version saved to '{output_path}'. "
#         f"Work autonomously and infer the required Python libraries. "
#         f"The dataset already has a safe schema, so do not rename, reorder, merge, split, or drop columns unless strictly necessary. "
#         f"Focus only on intra-column semantic consistency. "
#         f"For each column, inspect non-null values and infer the dominant representation. "
#         f"Ignore missing or non-informative values when inferring patterns. "
#         f"Normalize only high-confidence minority variants. "
#         f"Do not collapse distinct categories. "
#         f"Preserve uncertain or difform values and report them. "
#         f"Before saving, validate that the dataframe is non-empty and still loadable. "
#         f"The dataset must be loaded, processed, and saved within the same execution flow. "
#         f"All steps must run automatically without requiring any explicit invocation or entry point. "
#         f"If validation passes, write the output file immediately using to_csv. "
#         f"Do not check whether the output file already exists before writing it. "
#         f"The output file must always be created during execution if the dataframe is valid. "
#         f"Save the semantically refined dataset to '{output_path}' without index. "
#         + _findings_guidance(
#             findings_key,
#             "Include shape_before, shape_after, normalization_mappings, preserved_difform_values. "
#         )
#     )


# # ── Task 3: Merge ─────────────────────────────────────────────────────────────

# def _build_merge_prompt():
#     return (
#         f"You are a data integration agent responsible for combining two cleaned tabular datasets into a single unified dataset for downstream anomaly detection. "
#         f"Load the dataset at '{OUTPUT_DIR}/allarmi_clean.csv' and the dataset at '{OUTPUT_DIR}/tipologia_clean.csv' using pandas. "
#         f"Work autonomously and infer the required Python libraries. "
#         f"First, inspect both dataframes independently: print their shapes and column names. "
#         f"Then identify the common columns between the two dataframes and print them explicitly before proceeding. "
#         f"Merge the two dataframes on the common columns using an outer join to preserve all records from both sources. "
#         f"After merging, ensure that no duplicate columns remain in the result. "
#         f"Do not drop or rename any column unless it is a confirmed structural duplicate introduced by the merge. "
#         f"Do not perform any imputation, encoding, or transformation on the merged data. "
#         f"Preserve the original values and types from both sources as-is. "
#         f"Before saving, validate that the merged dataframe is non-empty and has unique column names. "
#         f"Do not save any output if those checks fail. "
#         f"Save the merged dataset to '{OUTPUT_DIR}/merged_data.csv' without index. "
#         f"Ensure that the dataset is loaded, processed, and saved within the same execution flow. "
#         f"Avoid defining execution entry points or structures that require explicit invocation. "
#         f"Assume that the code will be executed exactly as written, so all steps must run immediately. "
#         f"Print only a short summary with: shape of allarmi_clean, shape of tipologia_clean, common columns used for merge, final merged shape, and duplicate columns removed. "
#         + _findings_guidance(
#             "merge",
#             "Record shapes_allarmi_tipologia, common_columns, merged_shape, duplicate_columns_removed. "
#         )
#     )


# # ── Task 4: Group by route ────────────────────────────────────────────────────

# def _build_baseline_prompt():
#     return (
#         f"You are a data aggregation agent responsible for building a route-level summary dataset to be used as baseline input for downstream anomaly detection. "
#         f"Load the dataset at '{OUTPUT_DIR}/merged_data.csv'. "
#         f"Work autonomously and infer the required Python libraries. "
#         f"First, inspect the loaded dataframe: print its shape and column names before proceeding. "
#         f"Create a 'route' column by combining the values of 'areoporto_partenza' and 'areoporto_arrivo'. "
#         f"Do not drop the original airport columns after creating the route column. "
#         f"Build the aggregation strategy for every column except 'route'. "
#         f"Do not hardcode column names when defining the aggregation strategy. "
#         f"Determine whether a column is numeric using pandas-native type inspection, not numpy subtype checks. "
#         f"For numeric columns, aggregate by summing their values. "
#         f"For non-numeric columns, aggregate by taking the first observed value. "
#         f"Do not apply any transformation or normalization to the aggregated values. "
#         f"Group the dataframe by 'route' and aggregate all other columns according to the strategy above. "
#         f"After aggregation, ensure that 'route' is present as a standard column in the final dataframe and not only as an index. "
#         f"Before saving, validate that the resulting dataframe is non-empty and has unique column names. "
#         f"Do not save any output if those checks fail. "
#         f"Save the aggregated dataset to '{OUTPUT_DIR}/routes_summary.csv' without index. "
#         f"Ensure that the dataset is loaded, processed, and saved within the same execution flow. "
#         f"Avoid defining execution entry points or structures that require explicit invocation. "
#         f"Assume that the code will be executed exactly as written, so all steps must run immediately. "
#         f"Print only a short summary with: original merged shape, number of unique routes found, final routes_summary shape. "
#         + _findings_guidance(
#             "baseline_grouping",
#             "Store merged_shape, unique_routes, aggregation_strategy_summary (list of numeric/non_numeric columns), routes_summary_shape. "
#         )
#     )


# # ── Task 5: Baseline statistics ───────────────────────────────────────────────

# def _build_baseline_stats_prompt():
#     return (
#         f"You are a baseline statistics agent responsible for enriching an already aggregated route-level dataset with baseline features for downstream anomaly detection. "

#         f"Load the dataset at '{OUTPUT_DIR}/routes_summary.csv'. "
#         f"Work autonomously and infer the required Python libraries. "

#         f"If a findings JSON related to this dataset exists (same folder, filename ending with '_findings.json'), load it, print a short recap, and use it to guide semantic interpretation (e.g., column roles, numeric fields, warnings, distributions). "

#         f"The dataset is already aggregated (e.g., per route or similar grouping). "
#         f"Do not perform any additional grouping or aggregation. "

#         f"First inspect the dataframe and print its shape and column names. "

#         f"The dataset may contain column names in different languages (e.g., Italian) and with domain-specific terminology. "
#         f"You must infer semantic meaning from column names, distributions, and findings metadata, not from fixed keywords. "

#         f"Identify the main quantitative measure representing observed activity (e.g., counts, occurrences, alerts, volumes, flows, passengers, or investigated entities). "

#         f"To select this measure, follow these rules: "
#         f"- Prefer columns that are numeric or can be reliably converted to numeric. "
#         f"- Prefer columns with meaningful variability (not constant, not mostly zero). "
#         f"- Prefer columns representing aggregated quantities rather than identifiers or categories. "
#         f"- Exclude columns that are clearly descriptive, categorical, identifiers, codes, dates, locations, or textual explanations. "
#         f"- If findings metadata provides hints about measure columns, prioritize them. "

#         f"If multiple candidates exist, choose the one that best represents the main observed phenomenon of the dataset (e.g., traffic, alerts, occurrences). "
#         f"If no suitable column is found, raise an explicit error instead of forcing a weak choice. "

#         f"Once the measure is selected, convert it to numeric safely, preserving missing values where appropriate. "

#         f"Compute the global mean and global standard deviation of this measure across all rows. "
#         f"If the standard deviation is zero or missing, set it to 1 to avoid invalid scaling. "
#         f"If the mean is zero or missing, keep a safe fallback to avoid division errors. "

#         f"Add the following columns: "
#         f"- 'rolling_mean_alarms': global mean of the selected measure "
#         f"- 'rolling_std_alarms': global standard deviation "
#         f"- 'z_score': standardized deviation from the mean "
#         f"- 'ratio_to_baseline': ratio between the value and the mean "

#         f"Replace invalid values such as inf or -inf with 0. "

#         f"Before saving, validate that the dataframe is non-empty and still consistent. "
#         f"Do not save output if validation fails. "

#         f"Print only a concise summary including: selected measure column, reason for selection, global mean, global std, and final shape. "
#         f"Also print the top 10 rows sorted by z_score in descending order. "

#         f"Save the resulting dataset to '{OUTPUT_DIR}/baseline_data.csv' without index. "

#         f"Ensure all steps execute immediately without requiring explicit function calls. "

#         + _findings_guidance(
#             "baseline_stats",
#             "Store selected_metric_name, global_mean, global_std, baseline_shape, and top_rows_by_z_score (list of records with key identifiers and computed scores)."
#         )
#     )


# # ── Task 6: Outlier Detection ─────────────────────────────────────────────────

# def _build_outlier_prompt():
#     algo = DEFAULT_OUTLIER_ALGORITHM
#     contam = ISOLATION_FOREST_CONTAMINATION
#     neighbors = LOF_N_NEIGHBORS
#     zscore_t = ZSCORE_THRESHOLD

#     return (
#         f"You are an outlier detection agent operating on a route-level baseline dataset. "
#         f"Load the dataset at '{OUTPUT_DIR}/baseline_data.csv'. "
#         f"If a shared findings JSON exists at '{FINDINGS_JSON}', load it and reuse relevant information from previous steps "
#         f"(especially baseline statistics and column validation). Continue even if it is missing. "
#         f"Work autonomously and infer the required Python libraries. "
#         f"First inspect the dataset: print shape and column names. "
#         f"Ensure that the columns representing engineered features are valid, numeric, and usable for modeling. "
#         f"The expected features are 'allarmati', 'z_score', and 'ratio_to_baseline'. "
#         f"Coerce invalid values to numeric form and handle non-finite values safely so that the dataset is model-ready. "
#         f"Construct a feature matrix using exactly these three features, preserving row alignment with the original dataset. "
#         + (
#             f"Apply an Isolation Forest model using a contamination level of {contam} to detect anomalies. "
#             if algo == "IsolationForest" else
#             f"Apply a Local Outlier Factor model using {neighbors} neighbors and contamination {contam}. "
#             if algo == "LOF" else
#             f"Detect anomalies using a z-score threshold of {zscore_t} on the absolute value of z_score. "
#         )
#         + f"Store the result in a new boolean column named 'anomaly'. "
#         f"Do not drop any columns or rows. Preserve the full dataset. "
#         f"Print a short summary with: total number of rows and number of detected anomalies. "
#         f"Print the top 10 anomalous rows sorted by highest anomaly severity using z_score, "
#         f"displaying only 'route', 'allarmati', and 'z_score'. "
#         f"Before saving, validate that the dataframe is non-empty and consistent. "
#         f"If validation passes, save the dataset to '{OUTPUT_DIR}/outlier_results.csv' without index. "
#         f"The dataset must be loaded, processed, and saved within the same execution flow. "
#         f"All steps must run automatically without requiring explicit invocation. "
#         + _findings_guidance(
#             "outlier_detection",
#             "Store total_rows as int, anomaly_rows as int, algorithm_used as string, "
#             "feature_columns as list of strings, and top_anomalies as a list of plain Python dicts with route, allarmati, z_score. "
#         )
#     )


# # ── Task 7: Risk Profiling ────────────────────────────────────────────────────

# def _build_risk_prompt():
#     mult = ALERT_RATE_MULTIPLIER
#     return (
#         f"Load '{OUTPUT_DIR}/outlier_results.csv' with pandas. "
#         f"Import numpy as np. "
#         f"Filter only rows where anomaly is True. Print how many. "
#         f"If zero, save empty dataframe to '{OUTPUT_DIR}/risk_profiled.csv' and print 'No anomalies'. "
#         f"Otherwise: "
#         f"rule_route = anom['ratio_to_baseline'] > {mult}. "
#         f"rule_zscore_high = anom['z_score'].abs() > 8. "
#         f"rule_zscore_med = (anom['z_score'].abs() > 5) & (~rule_zscore_high). "
#         f"conditions = [rule_route & rule_zscore_high, rule_route | rule_zscore_high, rule_zscore_med]. "
#         f"choices = ['CRITICAL', 'HIGH', 'MEDIUM']. "
#         f"anom['risk_level'] = np.select(conditions, choices, default='LOW'). "
#         f"Print risk_level value_counts. "
#         f"Save full dataframe to '{OUTPUT_DIR}/risk_profiled.csv' without index."
#         + _findings_guidance(
#             "risk_profiling",
#             "Store anomaly_rows, risk_level_counts, rules_used (text). "
#         )
#     )


# # ── Task 8: Report ────────────────────────────────────────────────────────────
# # FIX: removed stray `s` after opening parenthesis

# def _build_report_prompt():
#     return (
#         f"Load '{OUTPUT_DIR}/risk_profiled.csv' with pandas. "
#         f"Sort by z_score descending and take top 5 rows. "
#         f"Import OpenAI from openai. "
#         f"Create client: client = OpenAI(base_url='{LM_STUDIO_BASE_URL}', api_key='{LM_STUDIO_API_KEY}'). "
#         f"For each row, call client.chat.completions.create("
#         f"model='{LM_STUDIO_MODEL}', "
#         f"messages=[{{'role':'user','content':'Explain this transit anomaly in 2 sentences: ' + str(row.to_dict())}}], "
#         f"max_tokens=150). "
#         f"Get response text from response.choices[0].message.content. "
#         f"Build a text report with header 'TRANSIT ANOMALY REPORT' and today's date. "
#         f"Save report to '{OUTPUT_DIR}/anomaly_report.txt'. "
#         f"Also save a JSON version with json.dump to '{OUTPUT_DIR}/anomaly_report.json'. "
#         f"Print the report."
#         + _findings_guidance(
#             "report_generation",
#             "Store top_anomalies_in_report (list), report_txt_path, report_json_path. "
#         )
#     )


# # ── _build_tasks(): single source of truth ────────────────────────────────────
# # FIX: replaces the three duplicate TASKS list definitions

# def _build_tasks():
#     return [
#         ("data_loading_allarmi",   _build_data_prompt_1()),
#         ("data_loading_tipologia", _build_data_prompt_2()),
#         ("semantic_allarmi", _build_semantic_normalization_prompt(
#             f"{OUTPUT_DIR}/allarmi_clean.csv",
#             f"{OUTPUT_DIR}/allarmi_semantic.csv",
#             "semantic_allarmi",
#         )),
#         ("semantic_tipologia", _build_semantic_normalization_prompt(
#             f"{OUTPUT_DIR}/tipologia_clean.csv",
#             f"{OUTPUT_DIR}/tipologia_semantic.csv",
#             "semantic_tipologia",
#         )),
#         ("merge",             _build_merge_prompt()),
#         ("baseline_grouping", _build_baseline_prompt()),
#         ("baseline_stats",    _build_baseline_stats_prompt()),
#         ("outlier_detection", _build_outlier_prompt()),
#         ("risk_profiling",    _build_risk_prompt()),
#         ("report_generation", _build_report_prompt()),
#     ]


# TASKS = _build_tasks()


# Multi-Agent System

3 agents: Supervisor (routing logic), Code_executor (direct LLM → REPL),
Validator (file check — no LLM needed).

Key design: NO tool calling. The LLM generates pure Python code,
and the code_executor runs it directly in a REPL.
- Includes rate limiting for free-tier APIs.
- Includes task caching: if a task's output file already exists, skip it.

In [40]:
import os
import time
from typing import Literal
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.types import Command
from openai import OpenAI


# Ensure output dir exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Direct LLM client (no LangChain, no tool calling)
_client = OpenAI(base_url=LM_STUDIO_BASE_URL, api_key=LM_STUDIO_API_KEY)
# NOTE: _repl is defined in the Tool REPL cell above — not re-instantiated here.

# Rate limiting
_SECONDS_BETWEEN_CALLS = 8  # 10 RPM = 1 every 6s, we use 8s for safety
_last_call_time = 0.0

# Verbose logging
VERBOSE = True  # Set to False once pipeline is stable

# Cache control
USE_CACHE = True


#  State
class AgentState(MessagesState):
    next: str
    current_task_index: int
    task_status: str
    retry_count: int


MAX_RETRIES = 4

# Maps each task name to the file it MUST produce.
EXPECTED_FILES = {
    "data_loading_allarmi":   os.path.join(OUTPUT_DIR, "allarmi_clean.csv"),
    "data_loading_tipologia": os.path.join(OUTPUT_DIR, "tipologia_clean.csv"),
    "semantic_allarmi":       os.path.join(OUTPUT_DIR, "allarmi_semantic.csv"),
    "semantic_tipologia":     os.path.join(OUTPUT_DIR, "tipologia_semantic.csv"),
    "merge":                  os.path.join(OUTPUT_DIR, "merged_data.csv"),
    "baseline_grouping":      os.path.join(OUTPUT_DIR, "routes_summary.csv"),
    "baseline_stats":         os.path.join(OUTPUT_DIR, "baseline_data.csv"),
    "outlier_detection":      os.path.join(OUTPUT_DIR, "outlier_results.csv"),
    "risk_profiling":         os.path.join(OUTPUT_DIR, "risk_profiled.csv"),
    "report_generation":      os.path.join(OUTPUT_DIR, "anomaly_report.txt"),
}


# Helper: check if task output already exists (cache)
def _task_is_cached(task_name: str) -> bool:
    if not USE_CACHE:
        return False
    filepath = EXPECTED_FILES.get(task_name, "")
    if not filepath:
        return False
    if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
        size = os.path.getsize(filepath)
        print(f"  [cache] '{task_name}' → {os.path.basename(filepath)} already exists ({size:,} bytes), skipping.")
        return True
    return False


# Helper: rate-limited LLM call
def _call_llm(messages: list[dict], max_tokens: int = 4096) -> str:
    global _last_call_time

    elapsed = time.time() - _last_call_time
    if elapsed < _SECONDS_BETWEEN_CALLS:
        wait = _SECONDS_BETWEEN_CALLS - elapsed
        print(f"  [rate_limit] Waiting {wait:.0f}s...")
        time.sleep(wait)

    for attempt in range(3):
        try:
            _last_call_time = time.time()
            response = _client.chat.completions.create(
                model=LM_STUDIO_MODEL,
                messages=messages,
                max_tokens=max_tokens,
                temperature=0.0,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e):
                backoff = 15 * (2 ** attempt)  # 15s, 30s, 60s
                print(f"  [rate_limit] 429 received, waiting {backoff}s (attempt {attempt+1}/3)...")
                time.sleep(backoff)
            else:
                return f"LLM_ERROR: {e}"

    return "LLM_ERROR: 429 — rate limit exceeded after 3 retries"


# Helper: clean code from LLM response
def _clean_code(raw: str) -> str:
    code = raw.strip()
    if code.startswith("```"):
        lines = code.split("\n")
        lines = lines[1:]  # drop opening fence
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]
        code = "\n".join(lines)
    return code.strip()


# Helper: ask LLM for code and run it
def _ask_and_run(task_prompt: str, retry_context: str = "") -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a Python data analyst. "
                "Reply with ONLY Python code, nothing else. "
                "No markdown, no backticks, no explanations. Just code."
            ),
        },
        {"role": "user", "content": task_prompt},
    ]

    if retry_context:
        messages.append({
            "role": "user",
            "content": f"Error: {retry_context}\nFix the code. Reply with ONLY the corrected Python code.",
        })

    raw = _call_llm(messages)

    if raw.startswith("LLM_ERROR"):
        return raw

    code = _clean_code(raw)

    if not code:
        return "LLM_ERROR: empty code response"

    if VERBOSE:
        print(f"\n  {'─'*40}")
        print(f"  GENERATED CODE:")
        print(f"  {'─'*40}")
        for i, line in enumerate(code.split("\n"), 1):
            print(f"  {i:3d} | {line}")
        print(f"  {'─'*40}\n")

    return _repl.run(code)


# Supervisor
def supervisor_node(state: AgentState) -> Command[Literal["code_executor", "validator", "__end__"]]:
    idx = state.get("current_task_index", 0)
    status = state.get("task_status", "pending")
    retries = state.get("retry_count", 0)

    if idx >= len(TASKS):
        return Command(goto="__end__", update={"next": "__end__"})

    task_name, task_prompt = TASKS[idx]

    if status == "pending" and _task_is_cached(task_name):
        next_idx = idx + 1
        if next_idx >= len(TASKS):
            return Command(goto="__end__", update={
                "current_task_index": next_idx, "task_status": "done",
            })
        return Command(
            goto="supervisor",
            update={
                "messages": [HumanMessage(content=f"Cached: {task_name}", name="supervisor")],
                "current_task_index": next_idx,
                "task_status": "pending",
                "retry_count": 0,
            },
        )

    if status in ("pending", "failed"):
        if status == "failed" and retries >= MAX_RETRIES:
            print(f"\n  SKIP '{task_name}' after {retries} retries\n")
            next_idx = idx + 1
            return Command(
                goto="__end__" if next_idx >= len(TASKS) else "supervisor",
                update={
                    "messages": [HumanMessage(content=f"Skipped {task_name}.", name="supervisor")],
                    "current_task_index": next_idx,
                    "task_status": "pending",
                    "retry_count": 0,
                },
            )
        return Command(
            goto="code_executor",
            update={
                "messages": [HumanMessage(content=task_prompt, name="supervisor")],
                "task_status": "executing",
            },
        )

    if status == "executing":
        expected = EXPECTED_FILES.get(task_name, "")
        return Command(
            goto="validator",
            update={
                "messages": [HumanMessage(content=expected, name="supervisor")],
                "task_status": "validating",
            },
        )

    if status == "validating":
        last = state["messages"][-1].content if state["messages"] else ""
        if last.startswith("APPROVED"):
            next_idx = idx + 1
            print(f"  ✓ Task '{task_name}' OK")
            if next_idx >= len(TASKS):
                return Command(goto="__end__", update={
                    "current_task_index": next_idx, "task_status": "done",
                })
            return Command(
                goto="supervisor",
                update={
                    "current_task_index": next_idx,
                    "task_status": "pending",
                    "retry_count": 0,
                },
            )
        else:
            return Command(
                goto="supervisor",
                update={"task_status": "failed", "retry_count": retries + 1},
            )

    return Command(goto="__end__", update={})


# Code Executor
def code_executor_node(state: AgentState) -> Command[Literal["supervisor"]]:
    idx = state.get("current_task_index", 0)
    retries = state.get("retry_count", 0)
# 
    if idx >= len(TASKS):
        return Command(
            update={"messages": [HumanMessage(content="All tasks done.", name="code_executor")]},
            goto="supervisor",
        )

    task_name, task_prompt = TASKS[idx]

    retry_context = ""
    if retries > 0:
        last_msg = state["messages"][-1].content if state["messages"] else ""
        if "Error" in last_msg or "REJECTED" in last_msg:
            retry_context = last_msg

    print(f"  [code_executor] Running '{task_name}' (attempt {retries + 1})...")
    result = _ask_and_run(task_prompt, retry_context)

    if len(result) > 2000:
        result = result[:2000] + "\n... [truncated]"

    print(f"  [code_executor] Output: {result[:300]}")

    return Command(
        update={"messages": [HumanMessage(content=result, name="code_executor")]},
        goto="supervisor",
    )


# Validator (pure Python, no LLM needed)
def validator_node(state: AgentState) -> Command[Literal["supervisor"]]:
    filepath = state["messages"][-1].content if state["messages"] else ""
    filepath = filepath.strip()

    if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
        size = os.path.getsize(filepath)
        msg = f"APPROVED: {filepath} ({size:,} bytes)"
        print(f"  [validator] {msg}")
    else:
        if VERBOSE and os.path.isdir(OUTPUT_DIR):
            files = os.listdir(OUTPUT_DIR)
            print(f"  [validator] Files in output dir: {files}")
        msg = f"REJECTED: {filepath} not found or empty"
        print(f"  [validator] {msg}")

    return Command(
        update={"messages": [HumanMessage(content=msg, name="validator")]},
        goto="supervisor",
    )


# Graph
def build_graph():
    g = StateGraph(AgentState)
    g.add_node("supervisor", supervisor_node)
    g.add_node("code_executor", code_executor_node)
    g.add_node("validator", validator_node)
    g.add_edge(START, "supervisor")
    return g.compile()


In [41]:
def llm_callable(prompt: str) -> str:
    response = _client.chat.completions.create(
        model=LM_STUDIO_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a data profiling assistant. "
                    "When asked, you return ONLY a valid JSON object — "
                    "no preamble, no markdown fences, no explanations outside the JSON. "
                    "Do not invent facts not supported by the provided profile."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_tokens=600,
        response_format={"type": "json_object"},
    )
    return response.choices[0].message.content.strip()


def run_pre_pipeline_column_profiling() -> dict:
    datasets_to_profile = [
        ("allarmi_raw", ALLARMI_CSV),
        ("tipologia_raw", TIPOLOGIA_CSV),
    ]

    profiles = {}

    for dataset_name, csv_path in datasets_to_profile:
        if not os.path.exists(csv_path):
            print(f"  [profiling] File not found, skipped: {csv_path}")
            continue

        # Read original column names just to display them
        try:
            original_cols = list(pd.read_csv(csv_path, nrows=0).columns)
            print(f"  [profiling] Running column profiling for {dataset_name} "
                  f"({len(original_cols)} columns: {', '.join(original_cols[:5])}{'...' if len(original_cols) > 5 else ''})")
        except Exception:
            print(f"  [profiling] Running column profiling for {dataset_name}...")

        profiles[dataset_name] = build_dataset_column_profiles_from_csv(
            csv_path=csv_path,
            rich_json_path=COLUMN_PROFILES_JSON,
            slim_json_path=CLEANING_PLAN_JSON,
            llm_callable=llm_callable,
            dataset_name=dataset_name,
            sample_size=DOMINANT_FORMAT_SAMPLE_SIZE,
        )
        print(f"  [profiling] Saved profile for {dataset_name}")

    return profiles

In [42]:
print("=" * 60)
print("  PRE-PIPELINE COLUMN PROFILING")
print("=" * 60)

profiles = run_pre_pipeline_column_profiling()

print()
print("Files written:")
if os.path.exists(COLUMN_PROFILES_JSON):
    print(f"  ✓ {COLUMN_PROFILES_JSON} ({os.path.getsize(COLUMN_PROFILES_JSON):,} bytes)")
else:
    print(f"  ✗ {COLUMN_PROFILES_JSON} (NOT created)")

if os.path.exists(CLEANING_PLAN_JSON):
    print(f"  ✓ {CLEANING_PLAN_JSON} ({os.path.getsize(CLEANING_PLAN_JSON):,} bytes)")
else:
    print(f"  ✗ {CLEANING_PLAN_JSON} (NOT created)")

  PRE-PIPELINE COLUMN PROFILING
  [profiling] Running column profiling for allarmi_raw (24 columns: OCCORRENZE, AREOPORTO_ARRIVO, AREOPORTO_PARTENZA, ANNO_PARTENZA, MESE_PARTENZA...)
  [profiling] Saved profile for allarmi_raw
  [profiling] Running column profiling for tipologia_raw (33 columns: NAZIONALITA, AREOPORTO_ARRIVO, AREOPORTO_PARTENZA, ANNO_PARTENZA, MESE_PARTENZA...)
  [profiling] Saved profile for tipologia_raw

Files written:
  ✓ /Users/matteo/Desktop/FBI-Agents-817541/output/column_profiles.json (90,258 bytes)
  ✓ /Users/matteo/Desktop/FBI-Agents-817541/output/cleaning_plan.json (28,872 bytes)


# Cleaning agent

In [43]:
# =============================================================================
# CLEANING AGENT — constants, thresholds, recovery vocabulary
# =============================================================================
# This block defines the CONTRACT of the cleaning agent:
#   - which escalation triggers cause an LLM call
#   - which recovery actions the LLM is allowed to choose from
#   - the shape of the per-column log entry returned in the final report
#
# The agent is a closed-loop orchestrator: the transformation_rule coming
# from the cleaning_plan is treated as IMMUTABLE input. The LLM is only
# consulted when the deterministic application of the rule fails one of
# the escalation thresholds below.

from dataclasses import dataclass, field, asdict
from typing import Optional, Any


# ── Escalation thresholds (deterministic) ────────────────────────────
# A trigger fires when ANY of these conditions holds after applying the
# rule + enforcing the expected_dtype on a column.

CLEANING_AGENT_THRESHOLDS = {
    "max_non_conforming_pct":   20.0,   # % of rows still non-conforming after the rule
    "max_new_null_pct":         50.0,   # % of NEW nulls introduced by the rule
    # the other two triggers are boolean (dtype cast exception, fully empty column)
}


# ── Closed vocabulary of recovery actions ────────────────────────────
# The LLM MUST choose exactly one of these. No free-form code, no new actions.

RECOVERY_ACTIONS = {
    "set_column_null",             # null the whole column
    "drop_column",                 # remove the column from the df
    "relax_dtype_to_string",       # keep raw values, skip the dtype cast
    "nullify_non_conforming",      # keep conforming rows, null the rest
    "retry_with_different_action", # re-apply with a different TRANSFORMATION_ACTION
    "accept_as_is",                # do nothing, log a warning
}


# ── Trigger names used in the report ─────────────────────────────────

TRIGGER_NON_CONFORMING    = "non_conforming_above_threshold"
TRIGGER_DTYPE_CAST_FAILED = "dtype_cast_failed"
TRIGGER_NEW_NULLS         = "new_nulls_above_threshold"
TRIGGER_EMPTY_COLUMN      = "column_fully_empty"


# ── Per-column report entry ──────────────────────────────────────────

@dataclass
class ColumnCleaningLog:
    column_name: str
    plan_action: str                        # the action from the plan (immutable)
    metrics_before: dict = field(default_factory=dict)
    metrics_after_rule: dict = field(default_factory=dict)
    trigger: Optional[str] = None           # which trigger fired, if any
    recovery_action: Optional[str] = None   # what the LLM chose
    recovery_params: dict = field(default_factory=dict)
    metrics_after_recovery: dict = field(default_factory=dict)
    llm_reasoning: Optional[str] = None
    error: Optional[str] = None             # any unrecoverable error
    status: str = "ok"                      # ok | escalated | failed | skipped

    def to_dict(self) -> dict:
        return asdict(self)

In [44]:
# =============================================================================
# CLEANING AGENT — per-column metrics
# =============================================================================
# Compute metrics on the CURRENT state of a pandas Series.
#
# NOTE on duplication with the rich profile:
# The rich profile already contains row_count, null_count, missing_percentage,
# n_unique_non_null, dtype, and non_conforming.share_pct for every column,
# but those numbers refer to the ORIGINAL df, BEFORE Phase 1 (column-name
# normalization, missing-token standardization, text lowercasing).
#
# The cleaning agent operates on the df AFTER Phase 1, so the profile's
# numbers are not a valid "before" reference to compare against the
# post-rule state. The trigger logic (new NaNs introduced by the rule,
# residual non-conforming %) needs a snapshot taken on the same df the
# rule is about to modify. Hence we recompute here.
#
# No sample values are ever returned: the agent is metrics-only.

def compute_column_metrics(
    series: pd.Series,
    dominant_label: Optional[str] = None,
) -> dict:
    """
    Compute deterministic metrics for a single column on its current state.

    Returns a dict with:
      - n_total                : number of rows
      - n_nulls                : number of NaN / pd.NA values
      - pct_null               : percentage of nulls over n_total
      - n_unique               : number of distinct non-null values
      - dtype                  : current pandas dtype as string
      - n_non_conforming       : number of non-null rows whose format label
                                  does NOT match dominant_label (0 if the
                                  dominant_label is unknown)
      - pct_non_conforming     : percentage of non-conforming rows over
                                  non-null rows (0.0 if no non-null rows)
    """
    n_total = int(len(series))
    n_nulls = int(series.isna().sum())
    n_non_null = n_total - n_nulls

    pct_null = round(float(n_nulls / n_total * 100), 2) if n_total else 0.0
    n_unique = int(series.dropna().nunique())

    if dominant_label is None or n_non_null == 0:
        n_non_conforming = 0
        pct_non_conforming = 0.0
    else:
        non_null = series.dropna()
        conforming_mask = non_null.map(
            lambda v: _value_format_label(v) == dominant_label
        )
        n_non_conforming = int((~conforming_mask).sum())
        pct_non_conforming = round(
            float(n_non_conforming / n_non_null * 100), 2
        )

    return {
        "n_total": n_total,
        "n_nulls": n_nulls,
        "pct_null": pct_null,
        "n_unique": n_unique,
        "dtype": str(series.dtype),
        "n_non_conforming": n_non_conforming,
        "pct_non_conforming": pct_non_conforming,
    }

In [45]:
# =============================================================================
# CLEANING AGENT — escalation trigger detector
# =============================================================================
# Pure function: given the metrics measured BEFORE and AFTER applying the
# plan's transformation_rule on a single column, plus any exception raised
# by enforce_expected_dtype, decide whether an escalation trigger has fired.
#
# Returns None if the deterministic pipeline succeeded for this column.
# Otherwise returns a tuple (trigger_name, reason) where:
#   - trigger_name : one of the TRIGGER_* constants defined in Cell 1,
#                    used for structured logging and branching.
#   - reason       : short human-readable string (English) embedded in the
#                    LLM prompt so the model knows what went wrong.
#
# Priority order when multiple conditions fire simultaneously:
#   1. dtype cast exception       (hard failure, always wins)
#   2. column fully empty         (nothing left to save)
#   3. new nulls above threshold  (the rule nuked too many values)
#   4. non-conforming above threshold (the rule did not normalize enough)

def detect_trigger(
    metrics_before: dict,
    metrics_after_rule: dict,
    dtype_error: Optional[str] = None,
    thresholds: dict = CLEANING_AGENT_THRESHOLDS,
) -> Optional[tuple]:
    """
    Evaluate the 4 escalation triggers in priority order.
    Returns (trigger_name, reason) or None.
    """
    # ── 1. Hard failure: enforce_expected_dtype raised ───────────────
    if dtype_error:
        return (
            TRIGGER_DTYPE_CAST_FAILED,
            f"enforce_expected_dtype failed: {dtype_error}",
        )

    n_total    = int(metrics_after_rule.get("n_total", 0))
    n_nulls_af = int(metrics_after_rule.get("n_nulls", 0))
    n_nulls_bf = int(metrics_before.get("n_nulls", 0))

    # ── 2. Column fully empty after the rule ─────────────────────────
    if n_total > 0 and n_nulls_af >= n_total:
        return (
            TRIGGER_EMPTY_COLUMN,
            "column is fully null after applying the rule",
        )

    # ── 3. New nulls introduced by the rule above threshold ──────────
    # We consider nulls created BY the rule, relative to the rows that
    # were non-null before. This prevents false positives when the column
    # was already mostly null going in.
    n_non_null_before = n_total - n_nulls_bf
    new_nulls = max(0, n_nulls_af - n_nulls_bf)
    if n_non_null_before > 0:
        pct_new_nulls = round(float(new_nulls / n_non_null_before * 100), 2)
        if pct_new_nulls > thresholds["max_new_null_pct"]:
            return (
                TRIGGER_NEW_NULLS,
                f"rule introduced {pct_new_nulls}% new nulls "
                f"(threshold {thresholds['max_new_null_pct']}%)",
            )

    # ── 4. Residual non-conforming share above threshold ─────────────
    pct_nc = float(metrics_after_rule.get("pct_non_conforming", 0.0))
    if pct_nc > thresholds["max_non_conforming_pct"]:
        return (
            TRIGGER_NON_CONFORMING,
            f"{pct_nc}% of non-null values still do not match "
            f"the dominant_format after the rule "
            f"(threshold {thresholds['max_non_conforming_pct']}%)",
        )

    return None

In [46]:
# =============================================================================
# CLEANING AGENT — recovery action dispatcher
# =============================================================================
# Closed vocabulary of 6 recovery actions the LLM is allowed to choose when
# an escalation trigger fires on a column. Each function:
#   - receives (df, column_name, params, col_plan)
#   - mutates / replaces the column (or drops it)
#   - returns (df, info_dict) where info_dict describes what was executed
#
# The LLM NEVER writes code. It only selects one of these action names and
# optionally provides simple params. This keeps the cleaning step fully
# reproducible and bounds the blast radius of any LLM mistake.


# ── 1. set_column_null ────────────────────────────────────────────────
def _recovery_set_column_null(df: pd.DataFrame, column: str, params: dict, col_plan: dict):
    df[column] = pd.NA
    return df, {"executed": "set_column_null"}


# ── 2. drop_column ────────────────────────────────────────────────────
def _recovery_drop_column(df: pd.DataFrame, column: str, params: dict, col_plan: dict):
    df = df.drop(columns=[column])
    return df, {"executed": "drop_column"}


# ── 3. relax_dtype_to_string ──────────────────────────────────────────
# Keep the values produced by the rule, but skip the expected_dtype cast
# and force the column to plain pandas "string". Useful when the rule
# extracted the right substrings but the cast to Int64/Float64 fails
# because of residual edge cases.
def _recovery_relax_dtype_to_string(df: pd.DataFrame, column: str, params: dict, col_plan: dict):
    df[column] = df[column].astype("string")
    return df, {"executed": "relax_dtype_to_string"}


# ── 4. nullify_non_conforming ─────────────────────────────────────────
# Keep the rows that already match dominant_format, set the rest to NA.
# This is the "salvage what works" option when the rule cannot fix
# everything but the conforming part is still valuable.
def _recovery_nullify_non_conforming(df: pd.DataFrame, column: str, params: dict, col_plan: dict):
    dominant_label = col_plan.get("dominant_format")
    series = df[column]
    if dominant_label is None:
        # No dominant format known: we cannot decide what is conforming.
        return df, {
            "executed": "nullify_non_conforming",
            "skipped": "no dominant_format in plan",
        }

    conforming_mask = _row_matches_format(series, dominant_label)
    target_mask = ~conforming_mask & series.notna()
    n_nulled = int(target_mask.sum())

    df.loc[target_mask, column] = pd.NA
    return df, {
        "executed": "nullify_non_conforming",
        "n_nulled": n_nulled,
    }


# ── 5. retry_with_different_action ────────────────────────────────────
# Re-apply the transformation engine with a DIFFERENT action chosen by
# the LLM from TRANSFORMATION_ACTIONS (the same closed set used by the
# profiling step). `params["new_rule"]` must be a dict with action + params.
def _recovery_retry_with_different_action(df: pd.DataFrame, column: str, params: dict, col_plan: dict):
    new_rule = params.get("new_rule") or {}
    action = new_rule.get("action")

    if action not in TRANSFORMATION_ACTIONS:
        return df, {
            "executed": "retry_with_different_action",
            "skipped": f"invalid or missing action '{action}'",
        }

    dominant_label = col_plan.get("dominant_format")
    # Build a well-formed rule dict compatible with apply_transformation_rule
    rule = {
        "action": action,
        "target_format": new_rule.get("target_format", dominant_label),
        "params": new_rule.get("params", {}) or {},
        "description": new_rule.get("description", "LLM-chosen retry action"),
    }

    df[column] = apply_transformation_rule(df[column], rule, dominant_label)
    return df, {
        "executed": "retry_with_different_action",
        "retry_rule": rule,
    }


# ── 6. accept_as_is ───────────────────────────────────────────────────
def _recovery_accept_as_is(df: pd.DataFrame, column: str, params: dict, col_plan: dict):
    return df, {"executed": "accept_as_is"}


# ── Registry ──────────────────────────────────────────────────────────
RECOVERY_DISPATCHER = {
    "set_column_null":             _recovery_set_column_null,
    "drop_column":                 _recovery_drop_column,
    "relax_dtype_to_string":       _recovery_relax_dtype_to_string,
    "nullify_non_conforming":      _recovery_nullify_non_conforming,
    "retry_with_different_action": _recovery_retry_with_different_action,
    "accept_as_is":                _recovery_accept_as_is,
}


def execute_recovery_action(
    df: pd.DataFrame,
    column: str,
    recovery_action: str,
    params: dict,
    col_plan: dict,
) -> tuple:
    """
    Execute a recovery action chosen by the LLM.
    Returns (df, info_dict). If the action name is not in the registry,
    returns the df unchanged and an error info dict.
    """
    fn = RECOVERY_DISPATCHER.get(recovery_action)
    if fn is None:
        return df, {
            "executed": None,
            "error": f"unknown recovery_action '{recovery_action}'",
        }
    return fn(df, column, params or {}, col_plan or {})

In [47]:
# =============================================================================
# CLEANING AGENT — LLM recovery prompt
# =============================================================================
# Build the prompt sent to the LLM when an escalation trigger fires on a
# column. The LLM is asked to choose ONE recovery action from the closed
# vocabulary and return a single JSON object. No sample values are included:
# the agent is metrics-only by design, so that it cannot second-guess the
# profiling agent's transformation_rule.

def _build_recovery_prompt(
    column_name: str,
    col_plan: dict,
    metrics_before: dict,
    metrics_after_rule: dict,
    trigger_name: str,
    trigger_reason: str,
) -> str:
    """
    Build the prompt string for a single column recovery decision.
    """
    plan_view = {
        "expected_dtype":      col_plan.get("expected_dtype"),
        "dominant_format":     col_plan.get("dominant_format"),
        "transformation_rule": col_plan.get("transformation_rule"),
    }

    return f"""
You are the recovery step of a deterministic data-cleaning pipeline.

A transformation_rule, decided earlier by a separate profiling agent,
has just been applied to ONE column. The deterministic post-application
checks have fired an escalation trigger, meaning the rule alone did not
produce an acceptable result. Your job is to choose ONE recovery action
from a closed vocabulary.

You MUST NOT second-guess the transformation_rule itself: it is an
immutable contract from the profiling step. You are only deciding how
to recover from the post-application failure.

==============================================================================
CLOSED VOCABULARY OF RECOVERY ACTIONS
==============================================================================
{sorted(RECOVERY_ACTIONS)}

Meaning of each recovery action:
  - set_column_null             : null the whole column
  - drop_column                 : remove the column from the dataframe
  - relax_dtype_to_string       : keep the rule's output, skip the dtype cast,
                                  force the column to plain pandas string
  - nullify_non_conforming      : keep rows matching dominant_format, null the rest
  - retry_with_different_action : re-apply the transformation engine with a
                                  DIFFERENT action from the profiling vocabulary.
                                  You must provide params.new_rule with keys
                                  action, params (optional), target_format (optional),
                                  description (optional).
                                  Allowed actions for new_rule.action:
                                  {sorted(TRANSFORMATION_ACTIONS)}
  - accept_as_is                : do nothing, accept the current state

==============================================================================
COLUMN NAME
==============================================================================
{column_name}

==============================================================================
PLAN FOR THIS COLUMN (immutable)
==============================================================================
{json.dumps(plan_view, ensure_ascii=False, indent=2)}

==============================================================================
METRICS BEFORE APPLYING THE RULE
==============================================================================
{json.dumps(metrics_before, ensure_ascii=False, indent=2)}

==============================================================================
METRICS AFTER APPLYING THE RULE
==============================================================================
{json.dumps(metrics_after_rule, ensure_ascii=False, indent=2)}

==============================================================================
TRIGGER THAT FIRED
==============================================================================
trigger_name: {trigger_name}
reason:       {trigger_reason}

==============================================================================
OUTPUT
==============================================================================
Return ONLY a JSON object with EXACTLY these keys, no preamble, no markdown:
{{
  "recovery_action": "<one of the recovery actions above>",
  "params":          {{...}},          // may be empty
  "reasoning":       "<1-2 sentences explaining the choice>"
}}

Guidance on choosing:
- If the trigger is dtype_cast_failed but pct_non_conforming is low,
  relax_dtype_to_string is usually the right call.
- If the column is almost entirely nulls after the rule, prefer drop_column
  or set_column_null.
- If pct_non_conforming is high but a different action would clearly work
  (for example parse_date when the rule tried coerce_numeric), use
  retry_with_different_action.
- If the conforming rows are valuable and you just want to discard the
  broken rows, use nullify_non_conforming.
- Use accept_as_is sparingly, only when the current state is actually acceptable.
""".strip()

In [48]:
# =============================================================================
# CLEANING AGENT — parse & validate the LLM recovery response
# =============================================================================
# The LLM's output is untrusted. This cell converts whatever comes back into
# a well-formed dict with:
#   - recovery_action : one of RECOVERY_ACTIONS, guaranteed
#   - params          : dict, possibly empty
#   - reasoning       : str, possibly empty
#   - parse_error     : optional string set when fallback is used
#
# If parsing fails or the chosen action is not in the vocabulary, the
# function returns a safe fallback of "accept_as_is" so the pipeline never
# crashes because of a malformed LLM reply.


def _parse_llm_recovery_response(response_text: str) -> dict:
    """Parse the LLM JSON reply, tolerating ```json fences and extra whitespace."""
    text = str(response_text).strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return json.loads(text)


def _validate_retry_rule(new_rule: dict) -> Optional[str]:
    """
    Validate the `new_rule` payload required by retry_with_different_action.
    Returns an error string or None if valid.
    """
    if not isinstance(new_rule, dict):
        return "new_rule must be a dict"
    action = new_rule.get("action")
    if action not in TRANSFORMATION_ACTIONS:
        return f"new_rule.action '{action}' is not in TRANSFORMATION_ACTIONS"
    params = new_rule.get("params", {})
    if params is not None and not isinstance(params, dict):
        return "new_rule.params must be a dict or null"
    return None


def parse_and_validate_recovery_response(response_text: str) -> dict:
    """
    Parse and validate the LLM recovery response.
    Always returns a well-formed dict; never raises.
    """
    fallback = {
        "recovery_action": "accept_as_is",
        "params":          {},
        "reasoning":       "",
        "parse_error":     None,
    }

    try:
        parsed = _parse_llm_recovery_response(response_text)
    except Exception as e:
        fallback["parse_error"] = f"json parse failed: {e}"
        return fallback

    if not isinstance(parsed, dict):
        fallback["parse_error"] = "response is not a JSON object"
        return fallback

    action = parsed.get("recovery_action")
    if action not in RECOVERY_ACTIONS:
        fallback["parse_error"] = (
            f"recovery_action '{action}' is not in RECOVERY_ACTIONS"
        )
        return fallback

    params = parsed.get("params") or {}
    if not isinstance(params, dict):
        fallback["parse_error"] = "params is not a dict"
        return fallback

    # Action-specific validation
    if action == "retry_with_different_action":
        err = _validate_retry_rule(params.get("new_rule"))
        if err:
            fallback["parse_error"] = f"retry_with_different_action invalid: {err}"
            return fallback

    reasoning = parsed.get("reasoning", "")
    if not isinstance(reasoning, str):
        reasoning = str(reasoning)

    return {
        "recovery_action": action,
        "params":          params,
        "reasoning":       reasoning.strip(),
        "parse_error":     None,
    }

In [49]:
# =============================================================================
# CLEANING AGENT — per-column closed-loop processing
# =============================================================================
# Processes a single column end-to-end:
#   1. measure metrics_before on the current df
#   2. apply the plan's transformation_rule deterministically
#   3. try to enforce expected_dtype (capturing any exception)
#   4. measure metrics_after_rule
#   5. detect escalation trigger
#   6. if a trigger fired: build prompt, call llm_callable, parse response,
#      execute the chosen recovery action, re-measure
#   7. return the updated df and a ColumnCleaningLog
#
# The df is passed and returned so that drop_column can actually remove it.
# metrics_before is RECOMPUTED here on the post-Phase-1 df, not read from
# the profile (see the note in the metrics cell for why).


def process_column_with_agent(
    df: pd.DataFrame,
    column: str,
    col_plan: dict,
    llm_callable: Callable[[str], str],
    thresholds: dict = CLEANING_AGENT_THRESHOLDS,
) -> tuple:
    """
    Run the full closed-loop cleaning on one column.
    Returns (df, ColumnCleaningLog).
    """
    log = ColumnCleaningLog(
        column_name=column,
        plan_action=(col_plan.get("transformation_rule") or {}).get("action", "none"),
    )

    dominant_label = col_plan.get("dominant_format")
    expected_dtype = col_plan.get("expected_dtype")
    rule = col_plan.get("transformation_rule") or {}

    # ── Step 1: metrics before ───────────────────────────────────────
    log.metrics_before = compute_column_metrics(df[column], dominant_label)

    # ── Step 2: apply the rule (deterministic) ───────────────────────
    df[column] = apply_transformation_rule(df[column], rule, dominant_label)

    # ── Step 3: enforce dtype, capturing any exception ───────────────
    dtype_error: Optional[str] = None
    try:
        df[column] = enforce_expected_dtype(df[column], expected_dtype)
    except Exception as e:
        dtype_error = f"{type(e).__name__}: {e}"

    # ── Step 4: metrics after rule ───────────────────────────────────
    log.metrics_after_rule = compute_column_metrics(df[column], dominant_label)

    # ── Step 5: trigger detection ────────────────────────────────────
    trigger = detect_trigger(
        metrics_before=log.metrics_before,
        metrics_after_rule=log.metrics_after_rule,
        dtype_error=dtype_error,
        thresholds=thresholds,
    )

    if trigger is None:
        log.status = "ok"
        return df, log

    trigger_name, trigger_reason = trigger
    log.trigger = trigger_name
    log.status = "escalated"

    # ── Step 6: ask the LLM for a recovery action ────────────────────
    prompt = _build_recovery_prompt(
        column_name=column,
        col_plan=col_plan,
        metrics_before=log.metrics_before,
        metrics_after_rule=log.metrics_after_rule,
        trigger_name=trigger_name,
        trigger_reason=trigger_reason,
    )

    try:
        raw = llm_callable(prompt)
    except Exception as e:
        log.error = f"llm_callable failed: {type(e).__name__}: {e}"
        log.status = "failed"
        return df, log

    decision = parse_and_validate_recovery_response(raw)
    log.recovery_action = decision["recovery_action"]
    log.recovery_params = decision["params"]
    log.llm_reasoning   = decision["reasoning"]
    if decision["parse_error"]:
        log.error = decision["parse_error"]

    # ── Step 7: execute the recovery action ──────────────────────────
    try:
        df, exec_info = execute_recovery_action(
            df=df,
            column=column,
            recovery_action=decision["recovery_action"],
            params=decision["params"],
            col_plan=col_plan,
        )
    except Exception as e:
        log.error = f"recovery execution failed: {type(e).__name__}: {e}"
        log.status = "failed"
        return df, log

    # ── Re-measure if the column still exists ────────────────────────
    # (drop_column removes it, so we check before touching df[column])
    if column in df.columns:
        log.metrics_after_recovery = compute_column_metrics(df[column], dominant_label)
    else:
        log.metrics_after_recovery = {"dropped": True}

    # Merge execution info (e.g. "skipped: ...") into the log error field
    # without overwriting an earlier parse_error.
    if "skipped" in exec_info and not log.error:
        log.error = f"recovery skipped: {exec_info['skipped']}"

    return df, log

In [50]:
# =============================================================================
# CLEANING AGENT — top-level orchestrator
# =============================================================================
# Public entry point of the agentic cleaning pipeline. Mirrors the shape
# of the deterministic clean_dataset function, with two differences:
#   - accepts an llm_callable used only for closed-loop recovery
#   - also writes a report JSON describing every per-column decision
#
# Pipeline stages:
#   Phase 1 (global, deterministic) : column-name normalization +
#                                     missing-token standardization +
#                                     text lowercasing
#   Phase 2 (per-column, agentic)   : for each column listed in the plan,
#                                     apply the rule, measure, escalate to
#                                     LLM if a trigger fires
#   Phase 3 (global, deterministic) : drop exact duplicate rows
#
# Columns present in the df but missing from the plan are skipped and
# logged with status='skipped'. Columns present in the plan but missing
# from the df (e.g. dropped mid-pipeline) are simply not processed.


def clean_dataset_with_agent(
    input_csv_path: str,
    output_csv_path: str,
    cleaning_plan_path: str,
    dataset_key: str,
    llm_callable: Callable[[str], str],
    report_json_path: Optional[str] = None,
    thresholds: dict = CLEANING_AGENT_THRESHOLDS,
    read_csv_kwargs: Optional[dict] = None,
) -> dict:
    """
    Run the full agentic cleaning pipeline on a single dataset.
    Returns the report dict and (optionally) writes it to report_json_path.
    """
    read_csv_kwargs = read_csv_kwargs or {}

    # ── Load ─────────────────────────────────────────────────────────
    df = pd.read_csv(input_csv_path, **read_csv_kwargs)
    shape_before = df.shape
    original_columns = list(df.columns)

    # ── Load slim cleaning plan ──────────────────────────────────────
    plan = load_cleaning_plan(cleaning_plan_path, dataset_key)

    # ── PHASE 1: global deterministic steps ──────────────────────────
    df = normalize_column_names(df)
    df = deduplicate_column_names(df)
    df = standardize_missing_values(df)
    df = normalize_text_values(df)

    # ── PHASE 2: per-column agentic processing ───────────────────────
    per_column_logs = []

    # Snapshot the column list BEFORE the loop, because drop_column
    # recovery may mutate df.columns during iteration.
    columns_to_process = list(df.columns)

    for col in columns_to_process:
        if col not in df.columns:
            # The column was dropped by a previous recovery action.
            continue

        col_plan = plan.get(col)
        if not col_plan:
            per_column_logs.append(ColumnCleaningLog(
                column_name=col,
                plan_action="none",
                status="skipped",
                error="no plan entry for this column",
            ))
            continue

        df, log = process_column_with_agent(
            df=df,
            column=col,
            col_plan=col_plan,
            llm_callable=llm_callable,
            thresholds=thresholds,
        )
        per_column_logs.append(log)

    # ── PHASE 3: remove duplicate rows ───────────────────────────────
    n_rows_before_dedup = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    n_duplicates_removed = n_rows_before_dedup - len(df)

    # ── Save cleaned CSV ─────────────────────────────────────────────
    os.makedirs(os.path.dirname(output_csv_path) or ".", exist_ok=True)
    df.to_csv(output_csv_path, index=False)

    # ── Build report ─────────────────────────────────────────────────
    status_counts = {"ok": 0, "escalated": 0, "failed": 0, "skipped": 0}
    for log in per_column_logs:
        status_counts[log.status] = status_counts.get(log.status, 0) + 1

    report = {
        "dataset_key":            dataset_key,
        "input_csv":              input_csv_path,
        "output_csv":             output_csv_path,
        "shape_before":           list(shape_before),
        "shape_after":            list(df.shape),
        "original_columns":       original_columns,
        "final_columns":          list(df.columns),
        "duplicates_removed":     int(n_duplicates_removed),
        "thresholds":             thresholds,
        "status_counts":          status_counts,
        "per_column_logs":        [log.to_dict() for log in per_column_logs],
    }

    # ── Optional: write report ───────────────────────────────────────
    if report_json_path:
        os.makedirs(os.path.dirname(report_json_path) or ".", exist_ok=True)
        with open(report_json_path, "w", encoding="utf-8") as f:
            json.dump(report, f, ensure_ascii=False, indent=2, default=str)

    return report

In [51]:
# =============================================================================
# CLEANING AGENT — run on all dataset keys defined in the cleaning_plan
# =============================================================================
# One agent invocation per dataset_key, sequentially. Each run writes:
#   - output/<key>_agent_clean.csv    the cleaned dataset
#   - output/<key>_agent_report.json  the per-column cleaning report
#
# Paths are declared explicitly here so that adding a new dataset tomorrow
# only requires appending one entry to DATASETS_TO_CLEAN.

DATASETS_TO_CLEAN = {
    "allarmi_raw": {
        "input_csv":   ALLARMI_CSV,
        "output_csv":  os.path.join(OUTPUT_DIR, "allarmi_agent_clean.csv"),
        "report_json": os.path.join(OUTPUT_DIR, "allarmi_agent_report.json"),
    },
    "tipologia_raw": {
        "input_csv":   TIPOLOGIA_CSV,
        "output_csv":  os.path.join(OUTPUT_DIR, "tipologia_agent_clean.csv"),
        "report_json": os.path.join(OUTPUT_DIR, "tipologia_agent_report.json"),
    },
}


all_reports = {}

for dataset_key, paths in DATASETS_TO_CLEAN.items():
    print("\n" + "=" * 70)
    print(f"CLEANING AGENT — dataset_key: {dataset_key}")
    print("=" * 70)

    report = clean_dataset_with_agent(
        input_csv_path     = paths["input_csv"],
        output_csv_path    = paths["output_csv"],
        cleaning_plan_path = CLEANING_PLAN_JSON,
        dataset_key        = dataset_key,
        llm_callable       = llm_callable,
        report_json_path   = paths["report_json"],
    )
    all_reports[dataset_key] = report

    # ── Per-dataset summary ──────────────────────────────────────────
    print(f"Shape before / after : {report['shape_before']} -> {report['shape_after']}")
    print(f"Duplicates removed   : {report['duplicates_removed']}")
    print(f"Status counts        : {report['status_counts']}")
    print(f"Output CSV           : {paths['output_csv']}")
    print(f"Report JSON          : {paths['report_json']}")

    # Per-column one-liner
    print("\nPer-column log:")
    for entry in report["per_column_logs"]:
        status = entry["status"]
        col    = entry["column_name"]
        plan_a = entry["plan_action"]
        trig   = entry.get("trigger") or "-"
        rec    = entry.get("recovery_action") or "-"
        err    = entry.get("error") or ""
        print(
            f"  [{status:9s}] {col:28s} "
            f"plan={plan_a:22s} trigger={trig:34s} recovery={rec:27s} {err}"
        )


# ── Global summary across datasets ───────────────────────────────────
print("\n" + "=" * 70)
print("GLOBAL SUMMARY")
print("=" * 70)
for dataset_key, report in all_reports.items():
    sc = report["status_counts"]
    print(
        f"  {dataset_key:16s}  "
        f"shape {report['shape_before']} -> {report['shape_after']}  "
        f"ok={sc['ok']}  escalated={sc['escalated']}  "
        f"failed={sc['failed']}  skipped={sc['skipped']}"
    )


CLEANING AGENT — dataset_key: allarmi_raw


/var/folders/5n/l0_5206d1s34bt79qbv4pxfr0000gq/T/ipykernel_46591/35952292.py:117: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed.loc[remaining_mask] = pd.to_datetime(
/var/folders/5n/l0_5206d1s34bt79qbv4pxfr0000gq/T/ipykernel_46591/35952292.py:100: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(series, errors="coerce")


Shape before / after : [5080, 24] -> [5029, 24]
Duplicates removed   : 51
Status counts        : {'ok': 21, 'escalated': 3, 'failed': 0, 'skipped': 0}
Output CSV           : /Users/matteo/Desktop/FBI-Agents-817541/output/allarmi_agent_clean.csv
Report JSON          : /Users/matteo/Desktop/FBI-Agents-817541/output/allarmi_agent_report.json

Per-column log:
  [ok       ] occorrenze                   plan=normalize_case         trigger=-                                  recovery=-                           
  [ok       ] areoporto_arrivo             plan=normalize_case         trigger=-                                  recovery=-                           
  [ok       ] areoporto_partenza           plan=normalize_case         trigger=-                                  recovery=-                           
  [ok       ] anno_partenza                plan=extract_pattern        trigger=-                                  recovery=-                           
  [ok       ] mese_partenza       

/var/folders/5n/l0_5206d1s34bt79qbv4pxfr0000gq/T/ipykernel_46591/35952292.py:117: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed.loc[remaining_mask] = pd.to_datetime(
/var/folders/5n/l0_5206d1s34bt79qbv4pxfr0000gq/T/ipykernel_46591/35952292.py:100: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(series, errors="coerce")


Shape before / after : [5095, 33] -> [5034, 33]
Duplicates removed   : 61
Status counts        : {'ok': 29, 'escalated': 4, 'failed': 0, 'skipped': 0}
Output CSV           : /Users/matteo/Desktop/FBI-Agents-817541/output/tipologia_agent_clean.csv
Report JSON          : /Users/matteo/Desktop/FBI-Agents-817541/output/tipologia_agent_report.json

Per-column log:
  [ok       ] nazionalita                  plan=map_to_canonical       trigger=-                                  recovery=-                           
  [ok       ] areoporto_arrivo             plan=normalize_case         trigger=-                                  recovery=-                           
  [ok       ] areoporto_partenza           plan=normalize_case         trigger=-                                  recovery=-                           
  [ok       ] anno_partenza                plan=extract_pattern        trigger=-                                  recovery=-                           
  [ok       ] mese_partenza   

/var/folders/5n/l0_5206d1s34bt79qbv4pxfr0000gq/T/ipykernel_46591/35952292.py:100: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(series, errors="coerce")


In [52]:
# Diagnostic cell: find which column crashes during Step 2 (apply_transformation_rule)

import traceback

df_diag = pd.read_csv(ALLARMI_CSV)
df_diag = normalize_column_names(df_diag)
df_diag = deduplicate_column_names(df_diag)
df_diag = standardize_missing_values(df_diag)
df_diag = normalize_text_values(df_diag)

plan_diag = load_cleaning_plan(CLEANING_PLAN_JSON, "allarmi_raw")

print("Trying apply_transformation_rule on each column:")
for col in df_diag.columns:
    col_plan = plan_diag.get(col, {})
    if not col_plan:
        print(f"  SKIP  {col} (no plan)")
        continue

    dominant_label = col_plan.get("dominant_format")
    expected_dtype = col_plan.get("expected_dtype")
    rule           = col_plan.get("transformation_rule") or {}
    action         = rule.get("action", "none")

    try:
        new_series = apply_transformation_rule(df_diag[col], rule, dominant_label)
        try:
            enforce_expected_dtype(new_series, expected_dtype)
            print(f"  OK         {col:28s} action={action:20s} -> {expected_dtype}")
        except Exception as e2:
            print(f"  FAIL_CAST  {col:28s} action={action:20s} -> {expected_dtype}  |  {type(e2).__name__}: {e2}")
    except Exception as e:
        print(f"  FAIL_RULE  {col:28s} action={action:20s}  |  {type(e).__name__}: {e}")
        traceback.print_exc()
        break

Trying apply_transformation_rule on each column:
  OK         occorrenze                   action=normalize_case       -> string
  OK         areoporto_arrivo             action=normalize_case       -> string
  OK         areoporto_partenza           action=normalize_case       -> string
  OK         anno_partenza                action=extract_pattern      -> Int64
  OK         mese_partenza                action=map_to_canonical     -> Int64
  OK         data_partenza                action=parse_date           -> datetime64[ns]
  OK         descr_aereoporto_arr         action=map_to_canonical     -> string
  OK         descr_aereoporto_part        action=normalize_case       -> string
  OK         citta_arr                    action=normalize_case       -> string
  OK         citta_partenza               action=normalize_case       -> string
  OK         codice_paese_arr             action=map_to_canonical     -> string
  OK         codice_paese_part            action=map_to_canonical

/var/folders/5n/l0_5206d1s34bt79qbv4pxfr0000gq/T/ipykernel_46591/35952292.py:117: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed.loc[remaining_mask] = pd.to_datetime(


In [53]:
# Diagnostic: confronta stato pre/post per le 3 colonne problematiche

import pandas as pd

# Carica il df post-Phase-1 (ricostruiscilo, non l'abbiamo salvato)
df_raw = pd.read_csv(ALLARMI_CSV)
df_raw = normalize_column_names(df_raw)
df_raw = deduplicate_column_names(df_raw)
df_raw = standardize_missing_values(df_raw)
df_raw = normalize_text_values(df_raw)

# Carica il df post-cleaning prodotto dall'agent
df_clean = pd.read_csv(os.path.join(OUTPUT_DIR, "allarmi_agent_clean.csv"))

cols_to_inspect = ["anno_partenza", "data_partenza", "descr_aereoporto_arr", "descr_aereoporto_part"]

for col in cols_to_inspect:
    print("=" * 70)
    print(f"COLUMN: {col}")
    print("=" * 70)
    print("Value counts BEFORE (post-Phase-1, pre-rule):")
    print(df_raw[col].value_counts(dropna=False).head(15))
    print()
    print("Value counts AFTER (in the cleaned CSV):")
    print(df_clean[col].value_counts(dropna=False).head(15))
    print()
    print(f"Null count BEFORE: {df_raw[col].isna().sum()}    AFTER: {df_clean[col].isna().sum()}")
    print()

COLUMN: anno_partenza
Value counts BEFORE (post-Phase-1, pre-rule):
anno_partenza
2024         4677
2023          257
24             52
anno 2024      52
2024.          42
Name: count, dtype: int64

Value counts AFTER (in the cleaned CSV):
anno_partenza
2024    4721
2023     256
24        52
Name: count, dtype: int64

Null count BEFORE: 0    AFTER: 0

COLUMN: data_partenza
Value counts BEFORE (post-Phase-1, pre-rule):
data_partenza
2024-02-04 06:10:00    7
2024-02-29 16:40:00    6
2024-01-31 18:00:00    6
2024-01-19 08:55:00    5
2024-01-22 13:00:00    5
2024-02-08 10:00:00    5
2024/02/26             5
2024-01-04 11:30:00    5
2024-01-06 08:00:00    5
2024-01-07 18:00:00    5
2024-02-18 06:50:00    5
2024-02-09 12:50:00    4
05.01.2024             4
2024/02/25             4
gen 06 2024            4
Name: count, dtype: int64

Value counts AFTER (in the cleaned CSV):
data_partenza
NaN                    301
2024-02-04 06:10:00      6
2024-01-31 18:00:00      6
2024-01-19 08:55:00      5